# Tutorial: G25 Part A - Standalone AdView Workspace, Full Download, fMRIPrep Confounds, EV Files, FEAT, and MRIcroGL

This notebook is written to run in a clean Neurodesk working directory without `git clone` or `git pull`.
It creates a local `g25/` folder, downloads the full anatomical files plus the AdView-specific functional BIDS imaging needed for preprocessing,
attempts `fMRIPrep` across the approved subject list when the environment supports it, extracts FEAT-ready `fMRIPrep` confounds automatically,
falls back to MCFLIRT motion confounds for the representative AdView run if `fMRIPrep` confounds are not available yet, converts AdView events to FSL EV files,
runs one representative FEAT model, and renders the result with MRIcroGL.


## Outline

1. Create or reuse a local `g25/` workspace and download full anatomical files plus AdView-specific functional imaging from OpenNeuro.
2. Inspect one representative anatomy file, AdView BOLD file, and AdView event table.
3. Attempt `fMRIPrep` across the approved subjects and record the status of each run.
4. Run `extract_fmriprep_confounds.py` automatically and summarize the written vs missing confounds.
5. Choose the representative FEAT confound source: `fMRIPrep` confounds if present, otherwise MCFLIRT motion confounds.
6. Convert all available AdView `events.tsv` files to FSL 3-column EV files.
7. Run one representative first-level FEAT model for AdView.
8. Visualize the FEAT result with MRIcroGL and embed the capture.


## How To Read This Annotated Version

This copy is for readers who want to understand the workflow, not just execute it.

- The long setup cell defines constants, paths, environment-variable overrides, and helper functions that all later steps reuse.
- Step 1 creates a standalone `g25/` workspace and downloads the minimum imaging needed for an AdView-first workflow.
- Steps 3 and 4 separate preprocessing status from confound extraction: the notebook first records whether `fMRIPrep` can run, then extracts selected confounds if outputs exist.
- Step 5 bridges preprocessing and modeling by choosing either `fMRIPrep` confounds or an MCFLIRT fallback.
- Steps 6 through 8 convert events to FEAT inputs, run a representative first-level model, and render a QC view of the result.

The comments added inside code cells focus on intent, assumptions, and side effects so the pipeline can be audited top-to-bottom.

In [ ]:
# Notebook bootstrap and shared utilities.
# This cell defines the constants, paths, helper functions, and environment-variable
# overrides that every later step depends on.

from __future__ import annotations

import gzip
import importlib.util
import json
import os
import shutil
import ssl
import struct
import subprocess
import sys
import textwrap
import urllib.request
from pathlib import Path

# Check the live kernel up front. The notebook assumes these libraries are already
# installed because later steps use them repeatedly for download manifests,
# plotting, NIfTI I/O, and table summaries.
required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "nibabel": "nibabel",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise RuntimeError(
        "This notebook expects the current Jupyter kernel to already provide: "
        + ", ".join(missing)
    )

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

# Dataset-wide constants and environment overrides.
# These values define which OpenNeuro snapshot to query, which subjects to treat
# as the AdView cohort, and whether Step 3 should attempt a real fMRIPrep run.
plt.rcParams["figure.dpi"] = 120
CTX = ssl._create_unverified_context()
OPENNEURO_API = "https://openneuro.org/crn/graphql"
DATASET_ID = "ds007486"
SNAPSHOT_TAG = "1.0.0"
APPROVED_ADVIEW_SUBJECTS_DEFAULT = ["sub-11028", "sub-11039", "sub-11450"]
APPROVED_SUBJECTS_DEFAULT = list(APPROVED_ADVIEW_SUBJECTS_DEFAULT)
REPRESENTATIVE_SUBJECT = os.environ.get("G25_REPRESENTATIVE_SUBJECT", "sub-11039").strip()
TASK = "adview"
RUN = 1
ECHO = 1
PROJECT_ROOT_OVERRIDE = os.environ.get("G25_PROJECT_ROOT", "").strip()
RUN_REAL_FMRIPREP = os.environ.get("G25_RUN_FMRIPREP", "1").strip().lower() not in {"0", "false", "no"}


def normalize_subjects(values: list[str], default: list[str]) -> list[str]:
    """Normalize subject identifiers from environment-variable input.

    Inputs:
    - values: raw tokens from G25_SUBJECTS, with or without the 'sub-' prefix.
    - default: fallback subject list when no override is supplied.

    Returns:
    - A normalized subject list with the 'sub-' prefix added where needed.

    Side effects:
    - None. This only standardizes user input for later download and processing steps.
    """
    if not values:
        return list(default)
    normalized = []
    for value in values:
        clean = value.strip()
        if not clean:
            continue
        if not clean.startswith("sub-"):
            clean = f"sub-{clean}"
        normalized.append(clean)
    return normalized


SUBJECT_OVERRIDE = os.environ.get("G25_SUBJECTS", "").split()
APPROVED_SUBJECTS = normalize_subjects(SUBJECT_OVERRIDE, APPROVED_SUBJECTS_DEFAULT)
APPROVED_ADVIEW_SUBJECTS = [subject for subject in APPROVED_SUBJECTS if subject in APPROVED_ADVIEW_SUBJECTS_DEFAULT]


WORKSPACE_README = """# G25

Standalone AdView workspace created by this tutorial notebook.
This folder is intentionally lightweight and is designed to be created directly in Neurodesk without cloning a repository.
"""
CODE_README = """# G25 Standalone Notebook Workspace

This folder was created by the AdView tutorial notebook.
The notebook writes the scripts it needs into `code/`, downloads the full anatomical files plus the AdView-specific functional files needed for preprocessing, runs `fMRIPrep` when the runtime supports it, extracts FEAT-ready confounds, converts AdView events to FSL EV files, and runs one representative FEAT model.
"""
TEMPLATE_ADVIEW_FSF = '\n# FEAT version number\nset fmri(version) 6.00\n\n# Are we in MELODIC?\nset fmri(inmelodic) 0\n\n# Analysis level\n# 1 : First-level analysis\n# 2 : Higher-level analysis\nset fmri(level) 1\n\n# Which stages to run\n# 0 : No first-level analysis (registration and/or group stats only)\n# 7 : Full first-level analysis\n# 1 : Pre-processing\n# 2 : Statistics\nset fmri(analysis) 7\n\n# Use relative filenames\nset fmri(relative_yn) 0\n\n# Balloon help\nset fmri(help_yn) 1\n\n# Run Featwatcher\nset fmri(featwatcher_yn) 0\n\n# Cleanup first-level standard-space images\nset fmri(sscleanup_yn) 0\n\n# Output directory\nset fmri(outputdir) "OUTPUT"\n\n# TR(s)\nset fmri(tr) 1.615000\n\n# Total volumes\nset fmri(npts) NVOLS\n\n# Delete volumes\nset fmri(ndelete) 0\n\n# Perfusion tag/control order\nset fmri(tagfirst) 1\n\n# Number of first-level analyses\nset fmri(multiple) 1\n\n# Higher-level input type\n# 1 : Inputs are lower-level FEAT directories\n# 2 : Inputs are cope images from FEAT directories\nset fmri(inputtype) 2\n\n# Carry out pre-stats processing?\nset fmri(filtering_yn) 1\n\n# Brain/background threshold, %\nset fmri(brain_thresh) 10\n\n# Critical z for design efficiency calculation\nset fmri(critical_z) 5.3\n\n# Noise level\nset fmri(noise) 0.66\n\n# Noise AR(1)\nset fmri(noisear) 0.34\n\n# Motion correction\n# 0 : None\n# 1 : MCFLIRT\nset fmri(mc) 0\n\n# Spin-history (currently obsolete)\nset fmri(sh_yn) 0\n\n# B0 fieldmap unwarping?\nset fmri(regunwarp_yn) 0\n\n# GDC Test\nset fmri(gdc) ""\n\n# EPI dwell time (ms)\nset fmri(dwell) 0.7\n\n# EPI TE (ms)\nset fmri(te) 35\n\n# % Signal loss threshold\nset fmri(signallossthresh) 10\n\n# Unwarp direction\nset fmri(unwarp_dir) y-\n\n# Slice timing correction\n# 0 : None\n# 1 : Regular up (0, 1, 2, 3, ...)\n# 2 : Regular down\n# 3 : Use slice order file\n# 4 : Use slice timings file\n# 5 : Interleaved (0, 2, 4 ... 1, 3, 5 ... )\nset fmri(st) 0\n\n# Slice timings file\nset fmri(st_file) ""\n\n# BET brain extraction\nset fmri(bet_yn) 1\n\n# Spatial smoothing FWHM (mm)\nset fmri(smooth) "SMOOTH"\n\n# Intensity normalization\nset fmri(norm_yn) 0\n\n# Perfusion subtraction\nset fmri(perfsub_yn) 0\n\n# Highpass temporal filtering\nset fmri(temphp_yn) 0\n\n# Lowpass temporal filtering\nset fmri(templp_yn) 0\n\n# MELODIC ICA data exploration\nset fmri(melodic_yn) 0\n\n# Carry out main stats?\nset fmri(stats_yn) 1\n\n# Carry out prewhitening?\nset fmri(prewhiten_yn) 1\n\n# Add motion parameters to model\n# 0 : No\n# 1 : Yes\nset fmri(motionevs) 0\nset fmri(motionevsbeta) ""\nset fmri(scriptevsbeta) ""\n\n# Robust outlier detection in FLAME?\nset fmri(robust_yn) 0\n\n# Higher-level modelling\n# 3 : Fixed effects\n# 0 : Mixed Effects: Simple OLS\n# 2 : Mixed Effects: FLAME 1\n# 1 : Mixed Effects: FLAME 1+2\nset fmri(mixed_yn) 2\n\n# Higher-level permutations\nset fmri(randomisePermutations) 5000\n\n# Number of EVs\nset fmri(evs_orig) 5\nset fmri(evs_real) 5\nset fmri(evs_vox) 0\n\n# Number of contrasts\nset fmri(ncon_orig) 7\nset fmri(ncon_real) 7\n\n# Number of F-tests\nset fmri(nftests_orig) 0\nset fmri(nftests_real) 0\n\n# Add constant column to design matrix? (obsolete)\nset fmri(constcol) 0\n\n# Carry out post-stats steps?\nset fmri(poststats_yn) 1\n\n# Pre-threshold masking?\nset fmri(threshmask) ""\n\n# Thresholding\n# 0 : None\n# 1 : Uncorrected\n# 2 : Voxel\n# 3 : Cluster\nset fmri(thresh) 3\n\n# P threshold\nset fmri(prob_thresh) 0.05\n\n# Z threshold\nset fmri(z_thresh) 3.1000000000000005\n\n# Z min/max for colour rendering\n# 0 : Use actual Z min/max\n# 1 : Use preset Z min/max\nset fmri(zdisplay) 0\n\n# Z min in colour rendering\nset fmri(zmin) 2\n\n# Z max in colour rendering\nset fmri(zmax) 8\n\n# Colour rendering type\n# 0 : Solid blobs\n# 1 : Transparent blobs\nset fmri(rendertype) 1\n\n# Background image for higher-level stats overlays\n# 1 : Mean highres\n# 2 : First highres\n# 3 : Mean functional\n# 4 : First functional\n# 5 : Standard space template\nset fmri(bgimage) 1\n\n# Create time series plots\nset fmri(tsplot_yn) 1\n\n# Registration to initial structural\nset fmri(reginitial_highres_yn) 0\n\n# Search space for registration to initial structural\n# 0   : No search\n# 90  : Normal search\n# 180 : Full search\nset fmri(reginitial_highres_search) 90\n\n# Degrees of Freedom for registration to initial structural\nset fmri(reginitial_highres_dof) 3\n\n# Registration to main structural\nset fmri(reghighres_yn) 0\n\n# Search space for registration to main structural\n# 0   : No search\n# 90  : Normal search\n# 180 : Full search\nset fmri(reghighres_search) 90\n\n# Degrees of Freedom for registration to main structural\nset fmri(reghighres_dof) BBR\n\n# Registration to standard image?\nset fmri(regstandard_yn) 0\n\n# Use alternate reference images?\nset fmri(alternateReference_yn) 0\n\n# Standard image\nset fmri(regstandard) "/usr/share/fsl/5.0/data/standard/MNI152_T1_2mm_brain"\n\n# Search space for registration to standard space\n# 0   : No search\n# 90  : Normal search\n# 180 : Full search\nset fmri(regstandard_search) 90\n\n# Degrees of Freedom for registration to standard space\nset fmri(regstandard_dof) 12\n\n# Do nonlinear registration from structural to standard space?\nset fmri(regstandard_nonlinear_yn) 0\n\n# Control nonlinear warp field resolution\nset fmri(regstandard_nonlinear_warpres) 10\n\n# High pass filter cutoff\nset fmri(paradigm_hp) 200\n\n# Total voxels\nset fmri(totalVoxels) 71101800\n\n\n# Number of lower-level copes feeding into higher-level analysis\nset fmri(ncopeinputs) 0\n\n# 4D AVW data or FEAT directory (1)\nset feat_files(1) "DATA"\n\n# Add confound EVs text file\nset fmri(confoundevs) 1\n\n# Confound EVs text file for analysis 1\nset confoundev_files(1) "CONFOUNDEVS"\n\n# EV 1 title\nset fmri(evtitle1) "basic"\n\n# Basic waveform shape (EV 1)\n# 0 : Square\n# 1 : Sinusoid\n# 2 : Custom (1 entry per volume)\n# 3 : Custom (3 column format)\n# 4 : Interaction\n# 10 : Empty (all zeros)\nset fmri(shape1) 3\n\n# Convolution (EV 1)\n# 0 : None\n# 1 : Gaussian\n# 2 : Gamma\n# 3 : Double-Gamma HRF\n# 4 : Gamma basis functions\n# 5 : Sine basis functions\n# 6 : FIR basis functions\n# 8 : Alternate Double-Gamma\nset fmri(convolve1) 3\n\n# Convolve phase (EV 1)\nset fmri(convolve_phase1) 0\n\n# Apply temporal filtering (EV 1)\nset fmri(tempfilt_yn1) 0\n\n# Add temporal derivative (EV 1)\nset fmri(deriv_yn1) 0\n\n# Custom EV file (EV 1)\nset fmri(custom1) "EVDIR_Every_day_products.txt"\n\n# Orthogonalise EV 1 wrt EV 0\nset fmri(ortho1.0) 0\n\n# Orthogonalise EV 1 wrt EV 1\nset fmri(ortho1.1) 0\n\n# Orthogonalise EV 1 wrt EV 2\nset fmri(ortho1.2) 0\n\n# Orthogonalise EV 1 wrt EV 3\nset fmri(ortho1.3) 0\n\n# Orthogonalise EV 1 wrt EV 4\nset fmri(ortho1.4) 0\n\n# Orthogonalise EV 1 wrt EV 5\nset fmri(ortho1.5) 0\n\n# EV 2 title\nset fmri(evtitle2) "gambling"\n\n# Basic waveform shape (EV 2)\n# 0 : Square\n# 1 : Sinusoid\n# 2 : Custom (1 entry per volume)\n# 3 : Custom (3 column format)\n# 4 : Interaction\n# 10 : Empty (all zeros)\nset fmri(shape2) 3\n\n# Convolution (EV 2)\n# 0 : None\n# 1 : Gaussian\n# 2 : Gamma\n# 3 : Double-Gamma HRF\n# 4 : Gamma basis functions\n# 5 : Sine basis functions\n# 6 : FIR basis functions\n# 8 : Alternate Double-Gamma\nset fmri(convolve2) 3\n\n# Convolve phase (EV 2)\nset fmri(convolve_phase2) 0\n\n# Apply temporal filtering (EV 2)\nset fmri(tempfilt_yn2) 0\n\n# Add temporal derivative (EV 2)\nset fmri(deriv_yn2) 0\n\n# Custom EV file (EV 2)\nset fmri(custom2) "EVDIR_Gambling.txt"\n\n# Orthogonalise EV 2 wrt EV 0\nset fmri(ortho2.0) 0\n\n# Orthogonalise EV 2 wrt EV 1\nset fmri(ortho2.1) 0\n\n# Orthogonalise EV 2 wrt EV 2\nset fmri(ortho2.2) 0\n\n# Orthogonalise EV 2 wrt EV 3\nset fmri(ortho2.3) 0\n\n# Orthogonalise EV 2 wrt EV 4\nset fmri(ortho2.4) 0\n\n# Orthogonalise EV 2 wrt EV 5\nset fmri(ortho2.5) 0\n\n# EV 3 title\nset fmri(evtitle3) "vmpfc"\n\n# Basic waveform shape (EV 3)\n# 0 : Square\n# 1 : Sinusoid\n# 2 : Custom (1 entry per volume)\n# 3 : Custom (3 column format)\n# 4 : Interaction\n# 10 : Empty (all zeros)\nset fmri(shape3) 3\n\n# Convolution (EV 3)\n# 0 : None\n# 1 : Gaussian\n# 2 : Gamma\n# 3 : Double-Gamma HRF\n# 4 : Gamma basis functions\n# 5 : Sine basis functions\n# 6 : FIR basis functions\n# 8 : Alternate Double-Gamma\nset fmri(convolve3) 3\n\n# Convolve phase (EV 3)\nset fmri(convolve_phase3) 0\n\n# Apply temporal filtering (EV 3)\nset fmri(tempfilt_yn3) 0\n\n# Add temporal derivative (EV 3)\nset fmri(deriv_yn3) 0\n\n# Custom EV file (EV 3)\nset fmri(custom3) "EVDIR_vmPFC.txt"\n\n# Orthogonalise EV 3 wrt EV 0\nset fmri(ortho3.0) 0\n\n# Orthogonalise EV 3 wrt EV 1\nset fmri(ortho3.1) 0\n\n# Orthogonalise EV 3 wrt EV 2\nset fmri(ortho3.2) 0\n\n# Orthogonalise EV 3 wrt EV 3\nset fmri(ortho3.3) 0\n\n# Orthogonalise EV 3 wrt EV 4\nset fmri(ortho3.4) 0\n\n# Orthogonalise EV 3 wrt EV 5\nset fmri(ortho3.5) 0\n\n# EV 4 title\nset fmri(evtitle4) "attn"\n\n# Basic waveform shape (EV 4)\n# 0 : Square\n# 1 : Sinusoid\n# 2 : Custom (1 entry per volume)\n# 3 : Custom (3 column format)\n# 4 : Interaction\n# 10 : Empty (all zeros)\nset fmri(shape4) 3\n\n# Convolution (EV 4)\n# 0 : None\n# 1 : Gaussian\n# 2 : Gamma\n# 3 : Double-Gamma HRF\n# 4 : Gamma basis functions\n# 5 : Sine basis functions\n# 6 : FIR basis functions\n# 8 : Alternate Double-Gamma\nset fmri(convolve4) 3\n\n# Convolve phase (EV 4)\nset fmri(convolve_phase4) 0\n\n# Apply temporal filtering (EV 4)\nset fmri(tempfilt_yn4) 0\n\n# Add temporal derivative (EV 4)\nset fmri(deriv_yn4) 0\n\n# Custom EV file (EV 4)\nset fmri(custom4) "EVDIR_attention_check.txt"\n\n# Orthogonalise EV 4 wrt EV 0\nset fmri(ortho4.0) 0\n\n# Orthogonalise EV 4 wrt EV 1\nset fmri(ortho4.1) 0\n\n# Orthogonalise EV 4 wrt EV 2\nset fmri(ortho4.2) 0\n\n# Orthogonalise EV 4 wrt EV 3\nset fmri(ortho4.3) 0\n\n# Orthogonalise EV 4 wrt EV 4\nset fmri(ortho4.4) 0\n\n# Orthogonalise EV 4 wrt EV 5\nset fmri(ortho4.5) 0\n\n# EV 5 title\nset fmri(evtitle5) "MISSED_TRIAL"\n\n# Basic waveform shape (EV 5)\n# 0 : Square\n# 1 : Sinusoid\n# 2 : Custom (1 entry per volume)\n# 3 : Custom (3 column format)\n# 4 : Interaction\n# 10 : Empty (all zeros)\nset fmri(shape5) EV_SHAPE\n\n# Convolution (EV 5)\n# 0 : None\n# 1 : Gaussian\n# 2 : Gamma\n# 3 : Double-Gamma HRF\n# 4 : Gamma basis functions\n# 5 : Sine basis functions\n# 6 : FIR basis functions\n# 8 : Alternate Double-Gamma\nset fmri(convolve5) 3\n\n# Convolve phase (EV 5)\nset fmri(convolve_phase5) 0\n\n# Apply temporal filtering (EV 5)\nset fmri(tempfilt_yn5) 0\n\n# Add temporal derivative (EV 5)\nset fmri(deriv_yn5) 0\n\n# Custom EV file (EV 5)\nset fmri(custom5) "EVDIR_missed_trial.txt"\n\n# Orthogonalise EV 5 wrt EV 0\nset fmri(ortho5.0) 0\n\n# Orthogonalise EV 5 wrt EV 1\nset fmri(ortho5.1) 0\n\n# Orthogonalise EV 5 wrt EV 2\nset fmri(ortho5.2) 0\n\n# Orthogonalise EV 5 wrt EV 3\nset fmri(ortho5.3) 0\n\n# Orthogonalise EV 5 wrt EV 4\nset fmri(ortho5.4) 0\n\n# Orthogonalise EV 5 wrt EV 5\nset fmri(ortho5.5) 0\n\n# Contrast & F-tests mode\n# real : control real EVs\n# orig : control original EVs\nset fmri(con_mode_old) orig\nset fmri(con_mode) orig\n\n# Display images for contrast_real 1\nset fmri(conpic_real.1) 1\n\n# Title for contrast_real 1\nset fmri(conname_real.1) "basic"\n\n# Real contrast_real vector 1 element 1\nset fmri(con_real1.1) 1\n\n# Real contrast_real vector 1 element 2\nset fmri(con_real1.2) 0\n\n# Real contrast_real vector 1 element 3\nset fmri(con_real1.3) 0\n\n# Real contrast_real vector 1 element 4\nset fmri(con_real1.4) 0\n\n# Real contrast_real vector 1 element 5\nset fmri(con_real1.5) 0\n\n# Display images for contrast_real 2\nset fmri(conpic_real.2) 1\n\n# Title for contrast_real 2\nset fmri(conname_real.2) "gambling"\n\n# Real contrast_real vector 2 element 1\nset fmri(con_real2.1) 0\n\n# Real contrast_real vector 2 element 2\nset fmri(con_real2.2) 1\n\n# Real contrast_real vector 2 element 3\nset fmri(con_real2.3) 0\n\n# Real contrast_real vector 2 element 4\nset fmri(con_real2.4) 0\n\n# Real contrast_real vector 2 element 5\nset fmri(con_real2.5) 0\n\n# Display images for contrast_real 3\nset fmri(conpic_real.3) 1\n\n# Title for contrast_real 3\nset fmri(conname_real.3) "vmPFC"\n\n# Real contrast_real vector 3 element 1\nset fmri(con_real3.1) 0\n\n# Real contrast_real vector 3 element 2\nset fmri(con_real3.2) 0\n\n# Real contrast_real vector 3 element 3\nset fmri(con_real3.3) 1\n\n# Real contrast_real vector 3 element 4\nset fmri(con_real3.4) 0\n\n# Real contrast_real vector 3 element 5\nset fmri(con_real3.5) 0\n\n# Display images for contrast_real 4\nset fmri(conpic_real.4) 1\n\n# Title for contrast_real 4\nset fmri(conname_real.4) "g > b"\n\n# Real contrast_real vector 4 element 1\nset fmri(con_real4.1) -1.0\n\n# Real contrast_real vector 4 element 2\nset fmri(con_real4.2) 1.0\n\n# Real contrast_real vector 4 element 3\nset fmri(con_real4.3) 0\n\n# Real contrast_real vector 4 element 4\nset fmri(con_real4.4) 0.0\n\n# Real contrast_real vector 4 element 5\nset fmri(con_real4.5) 0\n\n# Display images for contrast_real 5\nset fmri(conpic_real.5) 1\n\n# Title for contrast_real 5\nset fmri(conname_real.5) "b > vmPFC"\n\n# Real contrast_real vector 5 element 1\nset fmri(con_real5.1) 1.0\n\n# Real contrast_real vector 5 element 2\nset fmri(con_real5.2) 0.0\n\n# Real contrast_real vector 5 element 3\nset fmri(con_real5.3) -1.0\n\n# Real contrast_real vector 5 element 4\nset fmri(con_real5.4) 0\n\n# Real contrast_real vector 5 element 5\nset fmri(con_real5.5) 0.0\n\n# Display images for contrast_real 6\nset fmri(conpic_real.6) 1\n\n# Title for contrast_real 6\nset fmri(conname_real.6) "g > vmPFC"\n\n# Real contrast_real vector 6 element 1\nset fmri(con_real6.1) 0.0\n\n# Real contrast_real vector 6 element 2\nset fmri(con_real6.2) 1.0\n\n# Real contrast_real vector 6 element 3\nset fmri(con_real6.3) -1.0\n\n# Real contrast_real vector 6 element 4\nset fmri(con_real6.4) 0\n\n# Real contrast_real vector 6 element 5\nset fmri(con_real6.5) 0\n\n# Display images for contrast_real 7\nset fmri(conpic_real.7) 1\n\n# Title for contrast_real 7\nset fmri(conname_real.7) "g > (b + vmPFC)"\n\n# Real contrast_real vector 7 element 1\nset fmri(con_real7.1) -1.0\n\n# Real contrast_real vector 7 element 2\nset fmri(con_real7.2) 2.0\n\n# Real contrast_real vector 7 element 3\nset fmri(con_real7.3) -1.0\n\n# Real contrast_real vector 7 element 4\nset fmri(con_real7.4) 0\n\n# Real contrast_real vector 7 element 5\nset fmri(con_real7.5) 0\n\n# Display images for contrast_orig 1\nset fmri(conpic_orig.1) 1\n\n# Title for contrast_orig 1\nset fmri(conname_orig.1) "basic"\n\n# Real contrast_orig vector 1 element 1\nset fmri(con_orig1.1) 1\n\n# Real contrast_orig vector 1 element 2\nset fmri(con_orig1.2) 0\n\n# Real contrast_orig vector 1 element 3\nset fmri(con_orig1.3) 0\n\n# Real contrast_orig vector 1 element 4\nset fmri(con_orig1.4) 0\n\n# Real contrast_orig vector 1 element 5\nset fmri(con_orig1.5) 0\n\n# Display images for contrast_orig 2\nset fmri(conpic_orig.2) 1\n\n# Title for contrast_orig 2\nset fmri(conname_orig.2) "gambling"\n\n# Real contrast_orig vector 2 element 1\nset fmri(con_orig2.1) 0\n\n# Real contrast_orig vector 2 element 2\nset fmri(con_orig2.2) 1\n\n# Real contrast_orig vector 2 element 3\nset fmri(con_orig2.3) 0\n\n# Real contrast_orig vector 2 element 4\nset fmri(con_orig2.4) 0\n\n# Real contrast_orig vector 2 element 5\nset fmri(con_orig2.5) 0\n\n# Display images for contrast_orig 3\nset fmri(conpic_orig.3) 1\n\n# Title for contrast_orig 3\nset fmri(conname_orig.3) "vmPFC"\n\n# Real contrast_orig vector 3 element 1\nset fmri(con_orig3.1) 0\n\n# Real contrast_orig vector 3 element 2\nset fmri(con_orig3.2) 0\n\n# Real contrast_orig vector 3 element 3\nset fmri(con_orig3.3) 1\n\n# Real contrast_orig vector 3 element 4\nset fmri(con_orig3.4) 0\n\n# Real contrast_orig vector 3 element 5\nset fmri(con_orig3.5) 0\n\n# Display images for contrast_orig 4\nset fmri(conpic_orig.4) 1\n\n# Title for contrast_orig 4\nset fmri(conname_orig.4) "g > b"\n\n# Real contrast_orig vector 4 element 1\nset fmri(con_orig4.1) -1.0\n\n# Real contrast_orig vector 4 element 2\nset fmri(con_orig4.2) 1.0\n\n# Real contrast_orig vector 4 element 3\nset fmri(con_orig4.3) 0\n\n# Real contrast_orig vector 4 element 4\nset fmri(con_orig4.4) 0.0\n\n# Real contrast_orig vector 4 element 5\nset fmri(con_orig4.5) 0\n\n# Display images for contrast_orig 5\nset fmri(conpic_orig.5) 1\n\n# Title for contrast_orig 5\nset fmri(conname_orig.5) "b > vmPFC"\n\n# Real contrast_orig vector 5 element 1\nset fmri(con_orig5.1) 1.0\n\n# Real contrast_orig vector 5 element 2\nset fmri(con_orig5.2) 0.0\n\n# Real contrast_orig vector 5 element 3\nset fmri(con_orig5.3) -1.0\n\n# Real contrast_orig vector 5 element 4\nset fmri(con_orig5.4) 0\n\n# Real contrast_orig vector 5 element 5\nset fmri(con_orig5.5) 0.0\n\n# Display images for contrast_orig 6\nset fmri(conpic_orig.6) 1\n\n# Title for contrast_orig 6\nset fmri(conname_orig.6) "g > vmPFC"\n\n# Real contrast_orig vector 6 element 1\nset fmri(con_orig6.1) 0.0\n\n# Real contrast_orig vector 6 element 2\nset fmri(con_orig6.2) 1.0\n\n# Real contrast_orig vector 6 element 3\nset fmri(con_orig6.3) -1.0\n\n# Real contrast_orig vector 6 element 4\nset fmri(con_orig6.4) 0\n\n# Real contrast_orig vector 6 element 5\nset fmri(con_orig6.5) 0\n\n# Display images for contrast_orig 7\nset fmri(conpic_orig.7) 1\n\n# Title for contrast_orig 7\nset fmri(conname_orig.7) "g > (b + vmPFC)"\n\n# Real contrast_orig vector 7 element 1\nset fmri(con_orig7.1) -1.0\n\n# Real contrast_orig vector 7 element 2\nset fmri(con_orig7.2) 2.0\n\n# Real contrast_orig vector 7 element 3\nset fmri(con_orig7.3) -1.0\n\n# Real contrast_orig vector 7 element 4\nset fmri(con_orig7.4) 0\n\n# Real contrast_orig vector 7 element 5\nset fmri(con_orig7.5) 0\n\n# Contrast masking - use >0 instead of thresholding?\nset fmri(conmask_zerothresh_yn) 0\n\n# Mask real contrast/F-test 1 with real contrast/F-test 2?\nset fmri(conmask1_2) 0\n\n# Mask real contrast/F-test 1 with real contrast/F-test 3?\nset fmri(conmask1_3) 0\n\n# Mask real contrast/F-test 1 with real contrast/F-test 4?\nset fmri(conmask1_4) 0\n\n# Mask real contrast/F-test 1 with real contrast/F-test 5?\nset fmri(conmask1_5) 0\n\n# Mask real contrast/F-test 1 with real contrast/F-test 6?\nset fmri(conmask1_6) 0\n\n# Mask real contrast/F-test 1 with real contrast/F-test 7?\nset fmri(conmask1_7) 0\n\n# Mask real contrast/F-test 2 with real contrast/F-test 1?\nset fmri(conmask2_1) 0\n\n# Mask real contrast/F-test 2 with real contrast/F-test 3?\nset fmri(conmask2_3) 0\n\n# Mask real contrast/F-test 2 with real contrast/F-test 4?\nset fmri(conmask2_4) 0\n\n# Mask real contrast/F-test 2 with real contrast/F-test 5?\nset fmri(conmask2_5) 0\n\n# Mask real contrast/F-test 2 with real contrast/F-test 6?\nset fmri(conmask2_6) 0\n\n# Mask real contrast/F-test 2 with real contrast/F-test 7?\nset fmri(conmask2_7) 0\n\n# Mask real contrast/F-test 3 with real contrast/F-test 1?\nset fmri(conmask3_1) 0\n\n# Mask real contrast/F-test 3 with real contrast/F-test 2?\nset fmri(conmask3_2) 0\n\n# Mask real contrast/F-test 3 with real contrast/F-test 4?\nset fmri(conmask3_4) 0\n\n# Mask real contrast/F-test 3 with real contrast/F-test 5?\nset fmri(conmask3_5) 0\n\n# Mask real contrast/F-test 3 with real contrast/F-test 6?\nset fmri(conmask3_6) 0\n\n# Mask real contrast/F-test 3 with real contrast/F-test 7?\nset fmri(conmask3_7) 0\n\n# Mask real contrast/F-test 4 with real contrast/F-test 1?\nset fmri(conmask4_1) 0\n\n# Mask real contrast/F-test 4 with real contrast/F-test 2?\nset fmri(conmask4_2) 0\n\n# Mask real contrast/F-test 4 with real contrast/F-test 3?\nset fmri(conmask4_3) 0\n\n# Mask real contrast/F-test 4 with real contrast/F-test 5?\nset fmri(conmask4_5) 0\n\n# Mask real contrast/F-test 4 with real contrast/F-test 6?\nset fmri(conmask4_6) 0\n\n# Mask real contrast/F-test 4 with real contrast/F-test 7?\nset fmri(conmask4_7) 0\n\n# Mask real contrast/F-test 5 with real contrast/F-test 1?\nset fmri(conmask5_1) 0\n\n# Mask real contrast/F-test 5 with real contrast/F-test 2?\nset fmri(conmask5_2) 0\n\n# Mask real contrast/F-test 5 with real contrast/F-test 3?\nset fmri(conmask5_3) 0\n\n# Mask real contrast/F-test 5 with real contrast/F-test 4?\nset fmri(conmask5_4) 0\n\n# Mask real contrast/F-test 5 with real contrast/F-test 6?\nset fmri(conmask5_6) 0\n\n# Mask real contrast/F-test 5 with real contrast/F-test 7?\nset fmri(conmask5_7) 0\n\n# Mask real contrast/F-test 6 with real contrast/F-test 1?\nset fmri(conmask6_1) 0\n\n# Mask real contrast/F-test 6 with real contrast/F-test 2?\nset fmri(conmask6_2) 0\n\n# Mask real contrast/F-test 6 with real contrast/F-test 3?\nset fmri(conmask6_3) 0\n\n# Mask real contrast/F-test 6 with real contrast/F-test 4?\nset fmri(conmask6_4) 0\n\n# Mask real contrast/F-test 6 with real contrast/F-test 5?\nset fmri(conmask6_5) 0\n\n# Mask real contrast/F-test 6 with real contrast/F-test 7?\nset fmri(conmask6_7) 0\n\n# Mask real contrast/F-test 7 with real contrast/F-test 1?\nset fmri(conmask7_1) 0\n\n# Mask real contrast/F-test 7 with real contrast/F-test 2?\nset fmri(conmask7_2) 0\n\n# Mask real contrast/F-test 7 with real contrast/F-test 3?\nset fmri(conmask7_3) 0\n\n# Mask real contrast/F-test 7 with real contrast/F-test 4?\nset fmri(conmask7_4) 0\n\n# Mask real contrast/F-test 7 with real contrast/F-test 5?\nset fmri(conmask7_5) 0\n\n# Mask real contrast/F-test 7 with real contrast/F-test 6?\nset fmri(conmask7_6) 0\n\n# Do contrast masking at all?\nset fmri(conmask1_1) 0\n\n##########################################################\n# Now options that don\'t appear in the GUI\n\n# Alternative (to BETting) mask image\nset fmri(alternative_mask) ""\n\n# Initial structural space registration initialisation transform\nset fmri(init_initial_highres) ""\n\n# Structural space registration initialisation transform\nset fmri(init_highres) ""\n\n# Standard space registration initialisation transform\nset fmri(init_standard) ""\n\n# For full FEAT analysis: overwrite existing .feat output dir?\nset fmri(overwrite_yn) 1\n'
FMRIPREP_SH_TEXT = '#!/usr/bin/env bash\n\nset -euo pipefail\n\nquote_join() {\n  printf \' %q\' "$@"\n}\n\ndetect_module_loaded_fmriprep() {\n  if bash -lc "type ml >/dev/null 2>&1 && ml fmriprep >/dev/null 2>&1 && command -v fmriprep >/dev/null 2>&1"; then\n    echo "ml fmriprep"\n    return 0\n  fi\n  if bash -lc "type module >/dev/null 2>&1 && module load fmriprep >/dev/null 2>&1 && command -v fmriprep >/dev/null 2>&1"; then\n    echo "module load fmriprep"\n    return 0\n  fi\n  return 1\n}\n\nscriptdir="$( cd "$( dirname "${BASH_SOURCE[0]}" )" >/dev/null 2>&1 && pwd )"\nprojectdir="$(dirname "$scriptdir")"\nbidsdir="${BIDS_DIR:-${projectdir}/bids}"\noutdir="${FMRIPREP_OUTDIR:-${projectdir}/derivatives/fmriprep}"\nfilterfile="${BIDS_FILTER_FILE:-${projectdir}/code/fmriprep_config.json}"\n\nsubject="${1:-}"\nif [ -z "$subject" ]; then\n  echo "Usage: $(basename "$0") <subject-id-without-sub-prefix>"\n  echo "Example: $(basename "$0") 11039"\n  exit 1\nfi\n\nsubject="${subject#sub-}"\n\nsingularity_cmd="${SINGULARITY_CMD:-}"\nfmriprep_image="${FMRIPREP_IMAGE:-}"\nfmriprep_cmd="${FMRIPREP_CMD:-}"\nmodule_load_cmd="${FMRIPREP_MODULE_CMD:-}"\nscratch_base="${SCRATCH_BASE:-$HOME/scratch/g25}"\nscratchdir="${scratch_base}/fmriprep/sub-${subject}"\ndry_run="${DRY_RUN:-0}"\n\ntemplateflow_dir="${TEMPLATEFLOW_DIR:-}"
mplconfigdir_dir="${MPLCONFIGDIR_DIR:-}"
fs_license_file="${FS_LICENSE_FILE:-}"

if [ -z "$templateflow_dir" ] && [ -n "${TOOLS_DIR:-}" ]; then
  templateflow_dir="${TOOLS_DIR}/templateflow"
fi
if [ -z "$mplconfigdir_dir" ] && [ -n "${TOOLS_DIR:-}" ]; then
  mplconfigdir_dir="${TOOLS_DIR}/mplconfigdir"
fi
if [ -z "$fs_license_file" ]; then
  for candidate in \n    "$HOME/.license" \n    "$HOME/license.txt" \n    "$HOME/Desktop/license.txt"
  do
    if [ -f "$candidate" ]; then
      fs_license_file="$candidate"
      break
    fi
  done
fi
if [ -z "$fs_license_file" ] && [ -n "${TOOLS_DIR:-}" ]; then
  fs_license_file="${TOOLS_DIR}/licenses/fs_license.txt"
fi
if [ -z "$fmriprep_image" ] && [ -n "${TOOLS_DIR:-}" ]; then
  fmriprep_image="${TOOLS_DIR}/fmriprep-24.1.1.simg"
fi
\nimage_arg="${fmriprep_image:-<FMRIPREP_IMAGE>}"\n\nif [ -n "$fmriprep_cmd" ]; then\n  exec_mode="direct-cli"\nelif command -v fmriprep >/dev/null 2>&1; then\n  fmriprep_cmd="$(command -v fmriprep)"\n  exec_mode="direct-cli"\nelif [ -n "$module_load_cmd" ]; then\n  exec_mode="module-cli"\nelif module_load_cmd="$(detect_module_loaded_fmriprep)"; then\n  exec_mode="module-cli"\nelse\n  if [ -z "$singularity_cmd" ]; then\n    if command -v singularity >/dev/null 2>&1; then\n      singularity_cmd="singularity"\n    elif command -v apptainer >/dev/null 2>&1; then\n      singularity_cmd="apptainer"\n    fi\n  fi\n  if [ -n "$singularity_cmd" ] && command -v "$singularity_cmd" >/dev/null 2>&1; then\n    exec_mode="container"\n  else\n    exec_mode="unavailable"\n  fi\nfi\n\nif [ ! -d "${bidsdir}/sub-${subject}" ]; then\n  echo "ERROR: Missing BIDS directory ${bidsdir}/sub-${subject}"\n  exit 2\nfi\n\nmkdir -p "$outdir" "$scratchdir"\n\ncommon_args=(\n  participant --participant_label "$subject"\n  --stop-on-first-crash\n  --skip-bids-validation\n  --nthreads 14\n  --me-output-echos\n  --output-spaces MNI152NLin6Asym\n  --bids-filter-file "$filterfile"\n  --fs-no-reconall\n  -w "$scratchdir"\n)\n\nif [ -n "$fs_license_file" ]; then\n  common_args+=(--fs-license-file "$fs_license_file")\nfi\n\ncli_cmd=("$fmriprep_cmd" "$bidsdir" "$outdir" "${common_args[@]}")\n\ncontainer_cmd=(\n  "$singularity_cmd" run --cleanenv\n  -B "${projectdir}:/base"\n  -B "${scratchdir}:/scratch"\n)\n\nif [ -n "$templateflow_dir" ]; then\n  container_cmd+=(-B "${templateflow_dir}:/opt/templateflow")\nfi\nif [ -n "$mplconfigdir_dir" ]; then\n  container_cmd+=(-B "${mplconfigdir_dir}:/opt/mplconfigdir")\nfi\nif [ -n "$fs_license_file" ]; then\n  container_cmd+=(-B "$(dirname "$fs_license_file"):/opts")\nfi\n\ncontainer_cmd+=(\n  "$image_arg"\n  /base/bids /base/derivatives/fmriprep\n  participant --participant_label "$subject"\n  --stop-on-first-crash\n  --skip-bids-validation\n  --nthreads 14\n  --me-output-echos\n  --output-spaces MNI152NLin6Asym\n  --bids-filter-file /base/code/fmriprep_config.json\n  --fs-no-reconall\n  -w /scratch\n)\n\nif [ -n "$fs_license_file" ]; then\n  container_cmd+=(--fs-license-file "/opts/$(basename "$fs_license_file")")\nfi\n\necho "Subject: sub-${subject}"\necho "BIDS dir: ${bidsdir}"\necho "Output dir: ${outdir}"\necho "Scratch dir: ${scratchdir}"\necho "Filter file: ${filterfile}"\necho "Execution mode: ${exec_mode}"\necho "fMRIPrep command: ${fmriprep_cmd:-<unset>}"\necho "Module load command: ${module_load_cmd:-<unset>}"\necho "Container runtime: ${singularity_cmd:-<unset>}"\necho "TemplateFlow dir: ${templateflow_dir:-<unset>}"\necho "Matplotlib config dir: ${mplconfigdir_dir:-<unset>}"\necho "FreeSurfer license: ${fs_license_file:-<unset>}"\necho "fMRIPrep image: ${fmriprep_image:-<unset>}"\necho "Command:"\ncase "$exec_mode" in\n  direct-cli)\n    quote_join "${cli_cmd[@]}"\n    printf \'\\n\'\n    ;;\n  module-cli)\n    printf " bash -lc %q\\n" "${module_load_cmd} >/dev/null 2>&1 && exec$(quote_join "${cli_cmd[@]}")"\n    ;;\n  container)\n    quote_join "${container_cmd[@]}"\n    printf \'\\n\'\n    ;;\n  *)\n    echo " <unavailable>"\n    ;;\nesac\n\nif [ "$dry_run" = "1" ]; then\n  if [ "$exec_mode" = "unavailable" ]; then\n    echo "Missing prerequisites for a real run: no direct fMRIPrep command, no module-loaded fMRIPrep, and no Singularity/Apptainer runtime."\n  elif [ "$exec_mode" = "container" ]; then\n    missing=()\n    for required_var in templateflow_dir mplconfigdir_dir fs_license_file fmriprep_image; do\n      if [ -z "${!required_var}" ]; then\n        missing+=("$required_var")\n      fi\n    done\n    if [ "${#missing[@]}" -gt 0 ]; then\n      echo "Missing prerequisites for a real container run: ${missing[*]}"\n    fi\n  fi\n  echo "DRY_RUN=1, command not executed."\n  exit 0\nfi\n\ncase "$exec_mode" in\n  direct-cli)\n    [ -e "$filterfile" ] || { echo "ERROR: Missing required path: ${filterfile}"; exit 4; }\n    if [ -n "$templateflow_dir" ]; then export TEMPLATEFLOW_HOME="$templateflow_dir"; fi\n    if [ -n "$mplconfigdir_dir" ]; then export MPLCONFIGDIR="$mplconfigdir_dir"; fi\n    "${cli_cmd[@]}"\n    ;;\n  module-cli)\n    [ -e "$filterfile" ] || { echo "ERROR: Missing required path: ${filterfile}"; exit 4; }\n    shell_cmd="${module_load_cmd} >/dev/null 2>&1"\n    if [ -n "$templateflow_dir" ]; then\n      printf -v templateflow_q "%q" "$templateflow_dir"\n      shell_cmd="${shell_cmd} && export TEMPLATEFLOW_HOME=${templateflow_q}"\n    fi\n    if [ -n "$mplconfigdir_dir" ]; then\n      printf -v mplconfig_q "%q" "$mplconfigdir_dir"\n      shell_cmd="${shell_cmd} && export MPLCONFIGDIR=${mplconfig_q}"\n    fi\n    shell_cmd="${shell_cmd} && exec$(quote_join "${cli_cmd[@]}")"\n    bash -lc "$shell_cmd"\n    ;;\n  container)\n    if ! command -v "$singularity_cmd" >/dev/null 2>&1; then\n      echo "ERROR: Cannot find container runtime \'${singularity_cmd}\'."\n      echo "Set SINGULARITY_CMD, or rerun with DRY_RUN=1 for a preflight-only check."\n      exit 3\n    fi\n    for required_path in "$templateflow_dir" "$mplconfigdir_dir" "$fs_license_file" "$fmriprep_image" "$filterfile"; do\n      if [ ! -e "$required_path" ]; then\n        echo "ERROR: Missing required path: ${required_path}"\n        exit 4\n      fi\n    done\n    export SINGULARITYENV_TEMPLATEFLOW_HOME=/opt/templateflow\n    export SINGULARITYENV_MPLCONFIGDIR=/opt/mplconfigdir\n    "${container_cmd[@]}"\n    ;;\n  *)\n    echo "ERROR: Could not find a runnable fMRIPrep setup."\n    echo "Tried: direct \'fmriprep\' on PATH, module-loaded \'fmriprep\', and Singularity/Apptainer container modes."\n    echo "On Neurodesk, load fMRIPrep first with \'ml fmriprep\' before rerunning."\n    exit 3\n    ;;\nesac\n'
RUN_FMRIPREP_SH_TEXT = '#!/usr/bin/env bash\n\nset -euo pipefail\n\nscriptdir="$( cd "$( dirname "${BASH_SOURCE[0]}" )" >/dev/null 2>&1 && pwd )"\nprojectdir="$(dirname "$scriptdir")"\nscriptname="${scriptdir}/fmriprep.sh"\n\nsubjects=()\n\nif [ "$#" -gt 0 ]; then\n  subjects=("$@")\nelif [ -n "${SUBJECT_FILE:-}" ] && [ -f "${SUBJECT_FILE}" ]; then\n  while IFS= read -r sub; do\n    [ -n "$sub" ] || continue\n    subjects+=("$sub")\n  done < "${SUBJECT_FILE}"\nelif [ -f "${scriptdir}/sublist-new.txt" ]; then\n  while IFS= read -r sub; do\n    [ -n "$sub" ] || continue\n    subjects+=("$sub")\n  done < "${scriptdir}/sublist-new.txt"\nelse\n  while IFS= read -r subdir; do\n    subjects+=("${subdir##*/sub-}")\n  done < <(find "${projectdir}/bids" -maxdepth 1 -type d -name \'sub-*\' | sort)\nfi\n\nif [ "${#subjects[@]}" -eq 0 ]; then\n  echo "ERROR: No subjects found for fMRIPrep."\n  exit 1\nfi\n\nfor sub in "${subjects[@]}"; do\n  bash "$scriptname" "$sub"\ndone\n'
EXTRACT_CONFOUNDS_PY_TEXT = '#!/usr/bin/env python3\n\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport pandas as pd\n\n\nDEFAULT_TASKS = ["SST", "trust", "adview"]\nDEFAULT_RUNS = [1, 2, 3]\nTASK_RUNS = {\n    "SST": [1, 2, 3],\n    "trust": [1, 2, 3],\n    "adview": [1],\n}\nMOTION_COLS = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]\nQC_COLS = ["framewise_displacement", "dvars"]\nACOMPCOR_COLS = [f"a_comp_cor_{idx:02d}" for idx in range(6)]\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description="Extract a FEAT-friendly subset of fMRIPrep confounds across subjects/tasks/runs."\n    )\n    parser.add_argument(\n        "--project-root",\n        type=Path,\n        default=Path(__file__).resolve().parents[1],\n        help="Path to the g25 project root.",\n    )\n    parser.add_argument(\n        "--subjects",\n        nargs="*",\n        default=[],\n        help="Subjects to process, with or without the sub- prefix. Defaults to all subjects with local BIDS folders.",\n    )\n    parser.add_argument(\n        "--tasks",\n        nargs="*",\n        default=DEFAULT_TASKS,\n        help="Tasks to scan. Defaults to SST trust adview.",\n    )\n    parser.add_argument(\n        "--runs",\n        nargs="*",\n        type=int,\n        default=DEFAULT_RUNS,\n        help="Runs to scan. Defaults to 1 2 3.",\n    )\n    parser.add_argument(\n        "--outdir",\n        type=Path,\n        default=None,\n        help="Output directory for extracted confounds. Defaults to derivatives/fsl/confounds_fmriprep.",\n    )\n    return parser.parse_args()\n\n\ndef normalize_subjects(subjects: list[str], bids_root: Path) -> list[str]:\n    if subjects:\n        normalized = []\n        for subject in subjects:\n            clean = subject.removeprefix("sub-")\n            normalized.append(f"sub-{clean}")\n        return normalized\n    return sorted(path.name for path in bids_root.glob("sub-*") if path.is_dir())\n\n\ndef select_confound_columns(frame: pd.DataFrame) -> list[str]:\n    cols: list[str] = []\n    cols.extend([col for col in QC_COLS if col in frame.columns])\n    cols.extend([col for col in MOTION_COLS if col in frame.columns])\n    cols.extend([col for col in ACOMPCOR_COLS if col in frame.columns])\n    cols.extend(sorted(col for col in frame.columns if col.startswith("cosine")))\n    cols.extend(sorted(col for col in frame.columns if col.startswith("non_steady_state_outlier")))\n    return cols\n\n\ndef runs_for_task(task: str, requested_runs: list[int]) -> list[int]:\n    allowed = TASK_RUNS.get(task)\n    if not allowed:\n        return requested_runs\n    return [run for run in requested_runs if run in allowed]\n\n\ndef main() -> None:\n    args = parse_args()\n    project_root = args.project_root.resolve()\n    bids_root = project_root / "bids"\n    fmriprep_root = project_root / "derivatives" / "fmriprep"\n    outdir = args.outdir.resolve() if args.outdir else (project_root / "derivatives" / "fsl" / "confounds_fmriprep")\n    outdir.mkdir(parents=True, exist_ok=True)\n\n    subjects = normalize_subjects(args.subjects, bids_root)\n    records: list[dict[str, object]] = []\n\n    for subject in subjects:\n        for task in args.tasks:\n            for run in runs_for_task(task, args.runs):\n                filename = f"{subject}_task-{task}_run-{run}_part-mag_desc-confounds_timeseries.tsv"\n                source = fmriprep_root / subject / "func" / filename\n                record: dict[str, object] = {\n                    "subject": subject,\n                    "task": task,\n                    "run": run,\n                    "source_file": str(source),\n                }\n\n                if not source.exists():\n                    record["status"] = "missing"\n                    record["n_input_columns"] = 0\n                    record["n_output_columns"] = 0\n                    record["output_file"] = ""\n                    records.append(record)\n                    continue\n\n                frame = pd.read_csv(source, sep="\\t")\n                selected_cols = select_confound_columns(frame)\n                confounds = frame[selected_cols].fillna(0)\n\n                sub_outdir = outdir / subject\n                sub_outdir.mkdir(parents=True, exist_ok=True)\n                output = sub_outdir / f"{subject}_task-{task}_run-{run}_desc-fmriprepSelectedConfounds.tsv"\n                confounds.to_csv(output, sep="\\t", index=False, header=False)\n\n                record["status"] = "written"\n                record["n_input_columns"] = len(frame.columns)\n                record["n_output_columns"] = len(selected_cols)\n                record["selected_columns"] = ",".join(selected_cols)\n                record["output_file"] = str(output)\n                records.append(record)\n\n    manifest = pd.DataFrame(records).sort_values(["subject", "task", "run"]).reset_index(drop=True)\n    manifest_path = outdir / "confounds_manifest.csv"\n    manifest.to_csv(manifest_path, index=False)\n    print(manifest.to_string(index=False))\n    print()\n    print(f"Manifest saved to: {manifest_path}")\n\n\nif __name__ == "__main__":\n    main()\n'
FMRIPREP_CONFIG_JSON_TEXT = '{\n  "sbref": {"datatype": "func", "suffix": "sbref", "part": [null, "mag"]},\n  "bold": {"datatype": "func", "suffix": "bold", "part": [null, "mag"]}\n}\n'


def find_or_create_project_root(start: Path | None = None, override: str | Path | None = None) -> Path:
    """Locate an existing g25 workspace or create one locally.

    Inputs:
    - start: the directory from which upward and sibling searches begin.
    - override: an optional absolute or relative path from G25_PROJECT_ROOT.

    Returns:
    - A resolved Path pointing to the g25 project root used throughout the notebook.

    Side effects:
    - Creates a ./g25 directory when the notebook is run in a clean folder with no existing workspace.
    """
    candidates: list[Path] = []
    if override:
        candidates.append(Path(override).expanduser().resolve())

    start = Path.cwd() if start is None else Path(start)
    candidates.extend([start, *start.parents])

    for candidate in candidates:
        if candidate.name == "g25":
            return candidate.resolve()
        sibling_g25 = candidate / "g25"
        if sibling_g25.exists() and sibling_g25.is_dir():
            return sibling_g25.resolve()

    created = start / "g25"
    created.mkdir(parents=True, exist_ok=True)
    return created.resolve()


def run(cmd, cwd: Path | None = None, env: dict | None = None, check: bool = True) -> subprocess.CompletedProcess:
    """Execute a subprocess and capture stdout/stderr for notebook reporting.

    Inputs:
    - cmd: command list passed to subprocess.run.
    - cwd: optional working directory for the command.
    - env: optional environment variable overrides.
    - check: when True, raise a RuntimeError for non-zero exit codes.

    Returns:
    - subprocess.CompletedProcess so later cells can inspect outputs and status.

    Side effects:
    - Runs shell commands that may download data, invoke FSL/fMRIPrep, or write outputs.
    """
    run_env = dict(os.environ)
    if env:
        run_env.update(env)
    if cwd is not None:
        run_env["PWD"] = str(cwd)
        run_env["OLDPWD"] = str(cwd)
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=run_env,
        check=False,
        capture_output=True,
        text=True,
    )
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {cmd}\nSTDOUT\n{result.stdout}\nSTDERR\n{result.stderr}"
        )
    return result


def update_env_from_env0(env_dump: bytes) -> None:
    """Merge a null-delimited environment dump back into the current Python process.

    This is used after module-loader attempts so commands exposed by `ml` or `module load`
    become visible to later subprocess calls inside the same notebook kernel.
    """
    for entry in env_dump.split(b"\0"):
        if not entry or b"=" not in entry:
            continue
        key, value = entry.split(b"=", 1)
        os.environ[key.decode(errors="ignore")] = value.decode(errors="ignore")


def load_tool_module_if_needed(tool: str) -> tuple[bool, str | None]:
    """Try to expose a command-line tool through common HPC module systems.

    Inputs:
    - tool: module and command name, such as 'fsl' or 'fmriprep'.

    Returns:
    - (True, method) when the tool becomes available on PATH.
    - (False, None) when neither direct PATH lookup nor module loading succeeds.
    """
    if shutil.which(tool):
        return True, "already-on-path"
    attempts = [
        (f"ml {tool}", f"type ml >/dev/null 2>&1 && ml {tool}"),
        (f"module load {tool}", f"type module >/dev/null 2>&1 && module load {tool}"),
    ]
    for label, shell_cmd in attempts:
        result = subprocess.run(
            ["bash", "-lc", f"{shell_cmd} >/dev/null 2>&1 && env -0"],
            capture_output=True,
            check=False,
        )
        if result.returncode != 0 or not result.stdout:
            continue
        update_env_from_env0(result.stdout)
        if shutil.which(tool):
            return True, label
    return False, None


def load_fsl_module_if_needed() -> tuple[bool, str | None]:
    """Ensure FEAT-related FSL commands are available.

    This wrapper checks for mcflirt, feat, and fslnvols together because the notebook
    uses all three across the confounds and first-level modeling steps.
    """
    required_tools = ["mcflirt", "feat", "fslnvols"]
    if all(shutil.which(tool) for tool in required_tools):
        return True, "already-on-path"
    for tool in ["fsl"]:
        ok, label = load_tool_module_if_needed(tool)
        if ok and all(shutil.which(candidate) for candidate in required_tools):
            return True, label
    return False, None


def load_fmriprep_module_if_needed() -> tuple[bool, str | None]:
    """Ensure fMRIPrep is available either directly or through a module system."""
    if shutil.which("fmriprep"):
        return True, "already-on-path"
    return load_tool_module_if_needed("fmriprep")


def gql(query: str, variables: dict) -> dict:
    """Call the OpenNeuro GraphQL endpoint.

    Inputs:
    - query: GraphQL query string.
    - variables: GraphQL variables payload.

    Returns:
    - Parsed JSON response as a Python dictionary.
    """
    payload = json.dumps({"query": query, "variables": variables}).encode("utf-8")
    req = urllib.request.Request(
        OPENNEURO_API,
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, context=CTX) as resp:
        return json.load(resp)


def snapshot_files(tree: str | None = None) -> list[dict]:
    """List files for the dataset snapshot or a specific subtree within it.

    Inputs:
    - tree: optional snapshot subtree key.

    Returns:
    - A list of OpenNeuro file records containing keys, names, URLs, and sizes.
    """
    query = """
    query($datasetId: ID!, $tag: String!, $tree: String) {
      snapshot(datasetId: $datasetId, tag: $tag) {
        files(tree: $tree) {
          key
          filename
          directory
          urls
          size
        }
      }
    }
    """
    return gql(query, {"datasetId": DATASET_ID, "tag": SNAPSHOT_TAG, "tree": tree})["data"]["snapshot"]["files"]


def subject_directory_key(subject: str, root_rows: list[dict]) -> str:
    """Resolve the OpenNeuro tree key for a subject directory."""
    for row in root_rows:
        if row.get("directory") and row["filename"] == subject:
            return row["key"]
    raise FileNotFoundError(f"Missing subject directory for {subject}")


def child_directory_key(parent_key: str, dirname: str) -> str:
    """Resolve a child directory tree key beneath an OpenNeuro subject directory."""
    rows = snapshot_files(parent_key)
    for row in rows:
        if row.get("directory") and row["filename"] == dirname:
            return row["key"]
    raise FileNotFoundError(f"Missing child directory {dirname} under {parent_key}")


def is_valid_nifti_gz(path: Path) -> bool:
    """Perform a lightweight integrity check on a gzipped NIfTI file.

    The function verifies that the file is readable through gzip and that the header
    begins with the expected NIfTI header size. This protects later FSL steps from
    silently reusing corrupted cached downloads.
    """
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        with gzip.open(path, "rb") as src:
            header = src.read(352)
            for _chunk in iter(lambda: src.read(1024 * 1024), b""):
                pass
    except OSError:
        return False
    if len(header) < 352:
        return False
    sizeof_hdr_le = struct.unpack("<I", header[:4])[0]
    sizeof_hdr_be = struct.unpack(">I", header[:4])[0]
    return sizeof_hdr_le == 348 or sizeof_hdr_be == 348


def download_url(url: str, destination: Path, validate_nifti: bool = False) -> Path:
    """Download a file unless a valid local copy already exists.

    Inputs:
    - url: download source.
    - destination: local output path inside the standalone g25 workspace.
    - validate_nifti: when True, apply NIfTI integrity checks before accepting cache hits.

    Returns:
    - Path to the downloaded or reused local file.

    Side effects:
    - Creates parent directories and writes files to disk.
    """
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        if validate_nifti and not is_valid_nifti_gz(destination):
            destination.unlink()
        else:
            return destination

    with urllib.request.urlopen(url, context=CTX) as resp:
        destination.write_bytes(resp.read())

    if destination.stat().st_size == 0:
        raise RuntimeError(f"Downloaded empty file for {destination.name}")
    if validate_nifti and not is_valid_nifti_gz(destination):
        raise RuntimeError(f"Downloaded invalid gzipped NIfTI file: {destination}")
    return destination


def ensure_uncompressed_nifti(source_path: Path, destination_path: Path) -> Path:
    """Materialize an uncompressed .nii file for tools that read poorly from .nii.gz.

    Inputs:
    - source_path: compressed or uncompressed NIfTI source.
    - destination_path: runtime .nii path.

    Returns:
    - The uncompressed runtime file path.

    Side effects:
    - Writes or refreshes a runtime NIfTI used by MCFLIRT and FEAT.
    """
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    if destination_path.exists() and destination_path.stat().st_size > 0:
        if destination_path.stat().st_mtime >= source_path.stat().st_mtime:
            return destination_path
        destination_path.unlink()

    tmp_path = destination_path.with_name(destination_path.name + ".tmp")
    if tmp_path.exists():
        tmp_path.unlink()

    if source_path.name.endswith(".nii.gz"):
        with gzip.open(source_path, "rb") as src, tmp_path.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    else:
        shutil.copy2(source_path, tmp_path)

    if not tmp_path.exists() or tmp_path.stat().st_size == 0:
        raise RuntimeError(f"Failed to materialize runtime NIfTI for {source_path}")
    os.replace(tmp_path, destination_path)
    return destination_path


def write_file(path: Path, text: str, executable: bool = False) -> None:
    """Write a text asset into the standalone workspace and optionally mark it executable."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
    if executable:
        path.chmod(path.stat().st_mode | 0o111)


def patch_task_metadata(path: Path) -> None:
    """Replace placeholder task metadata so downstream tools and readers see a meaningful AdView task name."""
    if not path.exists():
        return
    payload = json.loads(path.read_text())
    if str(payload.get("TaskName", "")).startswith("TODO") or not payload.get("TaskName"):
        payload["TaskName"] = "Passive Ad Viewing"
    path.write_text(json.dumps(payload, indent=2) + "\n")


def plot_orthogonal_slices(volume: np.ndarray, title: str, out_path: Path, cmap: str = "gray") -> None:
    """Render sagittal, coronal, and axial views for quick notebook QC.

    Inputs:
    - volume: 3D numpy array or 3D slice from a 4D image.
    - title: figure title.
    - out_path: PNG destination.
    - cmap: matplotlib colormap name.
    """
    volume = np.asarray(volume)
    x, y, z = [dim // 2 for dim in volume.shape[:3]]
    vmax = float(np.percentile(volume, 99))
    vmin = float(np.percentile(volume, 1))

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(np.rot90(volume[x, :, :]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0].set_title("Sagittal")
    axes[1].imshow(np.rot90(volume[:, y, :]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[1].set_title("Coronal")
    axes[2].imshow(np.rot90(volume[:, :, z]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[2].set_title("Axial")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)


def derive_adview_trial_type(row: pd.Series) -> str:
    """Map raw AdView event rows onto the condition labels used for FEAT EV files.

    The side effects are indirect: later steps depend on these labels to create one
    EV text file per modeled condition.
    """
    response = str(row.get("attention_response", "")).strip()
    is_attention = str(row.get("is_attention_check", "")).strip().lower() in {"1", "true", "yes"}
    if response == "no_response":
        return "missed_trial"
    if is_attention:
        return "attention_check"
    return str(row["trial_type"])


def plot_adview_events(events: pd.DataFrame, count_path: Path, timeline_path: Path) -> None:
    """Write two QC figures that summarize the representative AdView event table.

    Outputs:
    - A bar plot of derived condition counts.
    - A timeline plot of event onsets by derived condition.
    """
    order = ["Every_day_products", "Gambling", "vmPFC", "attention_check", "missed_trial"]
    counts = events["derived_trial_type"].value_counts().reindex(order, fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 4))
    counts.plot(kind="bar", color=["#5B8FF9", "#F4664A", "#5D7092", "#61DDAA", "#9A60B4"], ax=ax)
    ax.set_title("AdView trial counts by derived event label")
    ax.set_xlabel("Derived trial type")
    ax.set_ylabel("Number of events")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    fig.savefig(count_path, dpi=150)
    plt.close(fig)

    y_map = {label: idx for idx, label in enumerate(order)}
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.scatter(events["onset"], [y_map[label] for label in events["derived_trial_type"]], s=28, c="#4C78A8")
    ax.set_yticks(list(y_map.values()))
    ax.set_yticklabels(order)
    ax.set_xlabel("Onset (s)")
    ax.set_title("AdView event timeline for the representative run")
    ax.grid(alpha=0.25, axis="x")
    fig.tight_layout()
    fig.savefig(timeline_path, dpi=150)
    plt.close(fig)


def convert_adview_events_to_evs(events_path: Path, ev_dir: Path, run_id: int = 1) -> list[dict[str, object]]:
    """Convert a BIDS AdView events.tsv file into FEAT-style 3-column EV files.

    Inputs:
    - events_path: source BIDS events.tsv file.
    - ev_dir: output directory for FEAT EV files.
    - run_id: run label used in output filenames.

    Returns:
    - A manifest list describing each EV file written.

    Side effects:
    - Creates one text file per modeled AdView condition.
    """
    ev_dir.mkdir(parents=True, exist_ok=True)
    frame = pd.read_csv(events_path, sep="\t").copy()
    frame["derived_trial_type"] = frame.apply(derive_adview_trial_type, axis=1)

    labels = ["Every_day_products", "Gambling", "vmPFC", "attention_check", "missed_trial"]
    manifest = []
    for label in labels:
        subset = frame.loc[frame["derived_trial_type"] == label, ["onset", "duration"]].copy()
        subset["amplitude"] = 1.0
        out_path = ev_dir / f"run-{run_id}_{label}.txt"
        if label == "missed_trial" and subset.empty:
            if out_path.exists():
                out_path.unlink()
            continue
        with out_path.open("w") as stream:
            for row in subset.itertuples(index=False):
                stream.write(f"{float(row.onset):.6f}\t{float(row.duration):.6f}\t1.0\n")
        manifest.append({"event": label, "rows": len(subset), "path": str(out_path)})
    return manifest


def find_first_command(candidates: list[str]) -> str | None:
    """Return the first executable command found on PATH from a preference list."""
    for candidate in candidates:
        command = shutil.which(candidate)
        if command:
            return command
    return None


def render_with_mricrogl(script_source: str, out_path: Path) -> tuple[Path, str, str | None]:
    """Run an MRIcroGL script and return the rendered image path plus tool information.

    Inputs:
    - script_source: MRIcroGL Python-style script text.
    - out_path: expected screenshot path.

    Returns:
    - (rendered_path, mricrogl_command_used, optional_mricron_command)
    """
    mricrogl_cmd = find_first_command(["mricrogl", "MRIcroGL"])
    mricron_cmd = find_first_command(["mricron", "MRIcron"])
    if mricrogl_cmd is None:
        raise FileNotFoundError(
            "Could not find mricrogl/MRIcroGL on PATH. "
            "This notebook is written for Neurodesk environments where MRIcroGL is available as a command-line tool."
        )

    script_path = RUNTIME_ROOT / "mricrogl_render.py"
    script_path.write_text(script_source)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    attempts = [
        [mricrogl_cmd, str(script_path)],
        [mricrogl_cmd, "-s", str(script_path)],
    ]
    last_error = None
    for cmd in attempts:
        result = run(cmd, cwd=RUNTIME_ROOT, check=False)
        if out_path.exists() and out_path.stat().st_size > 0:
            return out_path, mricrogl_cmd, mricron_cmd
        last_error = f"Tried {cmd}\nSTDOUT\n{result.stdout}\nSTDERR\n{result.stderr}"
    raise RuntimeError(f"MRIcroGL did not create {out_path}.\n{last_error}")


def choose_runtime_paths(project_root: Path, output_dir: Path) -> tuple[Path, Path, str]:
    """Choose where heavy runtime files should live.

    Returns:
    - runtime_root: top-level runtime directory.
    - runtime_project: project path visible to FSL/FEAT.
    - mode: short label describing whether the runtime is project-local or a tmp symlink.
    """
    if " " not in str(project_root):
        runtime_root = output_dir / "_runtime"
        runtime_root.mkdir(parents=True, exist_ok=True)
        return runtime_root, project_root, "project-local"

    runtime_root = Path("/tmp/g25_adview_runtime")
    runtime_root.mkdir(parents=True, exist_ok=True)
    runtime_project = runtime_root / project_root.name
    if runtime_project.exists() or runtime_project.is_symlink():
        if runtime_project.resolve() != project_root.resolve():
            runtime_project.unlink()
    if not runtime_project.exists():
        runtime_project.symlink_to(project_root, target_is_directory=True)
    return runtime_root, runtime_project, "tmp-symlink"


def parse_stdout_value(stdout: str, key: str) -> str:
    """Extract a simple 'Key: value' field from command stdout for status reporting."""
    prefix = f"{key}:"
    for line in stdout.splitlines():
        if line.startswith(prefix):
            return line.split(":", 1)[1].strip()
    return ""


# Materialize the standalone g25 workspace and the standard subdirectories that
# the later download, confound, EV, FEAT, and MRIcroGL steps expect to find.
PROJECT_ROOT = find_or_create_project_root(override=PROJECT_ROOT_OVERRIDE)
BIDS_ROOT = PROJECT_ROOT / "bids"
CODE_DIR = PROJECT_ROOT / "code"
TEMPLATE_DIR = PROJECT_ROOT / "templates"
DERIV_ROOT = PROJECT_ROOT / "derivatives"
FMRIPREP_OUTDIR = DERIV_ROOT / "fmriprep"
CONFOUNDS_FMRIPREP_DIR = DERIV_ROOT / "fsl" / "confounds_fmriprep"
EV_ROOT = DERIV_ROOT / "fsl" / "EVfiles"
OUTPUT_DIR = PROJECT_ROOT / "output" / "jupyter-notebook"
CACHE_DIR = OUTPUT_DIR / "_cache"
FIG_DIR = OUTPUT_DIR / "figures"
FEAT_DEMO_DIR = OUTPUT_DIR / "feat-demo"
CONF_DIR = FEAT_DEMO_DIR / "confounds"
FEAT_FIG_DIR = FEAT_DEMO_DIR / "figures"
WORKSPACE_SUMMARY_PATH = OUTPUT_DIR / "g25-adview-workspace-summary.json"
DOWNLOAD_MANIFEST_PATH = OUTPUT_DIR / "g25-approved-download-manifest.csv"
INVENTORY_PATH = OUTPUT_DIR / "g25-approved-subject-inventory.csv"
FMRIPREP_STATUS_PATH = OUTPUT_DIR / "g25-fmriprep-run-status.csv"
ADVIEW_TASK_JSON = BIDS_ROOT / "task-adview_bold.json"
ADVIEW_TEMPLATE_PATH = TEMPLATE_DIR / "L1_task-adview_model-1_type-act.fsf"
FMRIPREP_SCRIPT_PATH = CODE_DIR / "fmriprep.sh"
RUN_FMRIPREP_SCRIPT_PATH = CODE_DIR / "run_fmriprep.sh"
EXTRACT_CONFOUNDS_PATH = CODE_DIR / "extract_fmriprep_confounds.py"
FMRIPREP_CONFIG_PATH = CODE_DIR / "fmriprep_config.json"

for path in [
    BIDS_ROOT,
    CODE_DIR,
    TEMPLATE_DIR,
    FMRIPREP_OUTDIR,
    CONFOUNDS_FMRIPREP_DIR,
    EV_ROOT,
    OUTPUT_DIR,
    CACHE_DIR,
    FIG_DIR,
    FEAT_DEMO_DIR,
    CONF_DIR,
    FEAT_FIG_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

write_file(PROJECT_ROOT / "README.md", WORKSPACE_README)
write_file(CODE_DIR / "README.md", CODE_README)
write_file(ADVIEW_TEMPLATE_PATH, TEMPLATE_ADVIEW_FSF)
write_file(FMRIPREP_SCRIPT_PATH, FMRIPREP_SH_TEXT, executable=True)
write_file(RUN_FMRIPREP_SCRIPT_PATH, RUN_FMRIPREP_SH_TEXT, executable=True)
write_file(EXTRACT_CONFOUNDS_PATH, EXTRACT_CONFOUNDS_PY_TEXT, executable=True)
write_file(FMRIPREP_CONFIG_PATH, FMRIPREP_CONFIG_JSON_TEXT)

RUNTIME_ROOT, RUNTIME_PROJECT, RUNTIME_MODE = choose_runtime_paths(PROJECT_ROOT, OUTPUT_DIR)
FSL_READY, FSL_LOAD_METHOD = load_fsl_module_if_needed()
FMRIPREP_READY, FMRIPREP_LOAD_METHOD = load_fmriprep_module_if_needed()
MGL_CMD = find_first_command(["mricrogl", "MRIcroGL"])
MCRON_CMD = find_first_command(["mricron", "MRIcron"])

print(f"Project root: {PROJECT_ROOT}")
print(f"BIDS root: {BIDS_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Runtime root: {RUNTIME_ROOT}")
print(f"Runtime mode: {RUNTIME_MODE}")
print(f"Approved subjects: {APPROVED_SUBJECTS}")
print(f"Approved AdView subjects: {APPROVED_ADVIEW_SUBJECTS}")
print(f"Attempt real fMRIPrep runs: {RUN_REAL_FMRIPREP}")
print(f"FSL available: {FSL_READY}")
print(f"FSL load method: {FSL_LOAD_METHOD}")
print(f"fMRIPrep available: {FMRIPREP_READY}")
print(f"fMRIPrep load method: {FMRIPREP_LOAD_METHOD}")
print(f"mcflirt command: {find_first_command(['mcflirt'])}")
print(f"feat command: {find_first_command(['feat'])}")
print(f"fmriprep command: {find_first_command(['fmriprep'])}")
print(f"MRIcroGL command: {MGL_CMD}")
print(f"MRIcron command: {MCRON_CMD}")


## Step 1 - Create a standalone `g25/` workspace and download the anatomical files plus AdView-specific functional imaging

Purpose:
- The later `fMRIPrep` and confounds steps need actual BIDS imaging, not just event files.
- This step creates the minimal `g25/` folder structure and downloads the full `anat/` contents plus the AdView-specific `func/` files for the approved subject list directly from OpenNeuro.

What the code is doing:
- Query the `ds007486` `v1.0.0` snapshot.
- Download the root BIDS metadata files.
- Download every anatomical file for the approved subject list.
- Download only the AdView functional files needed for preprocessing, event conversion, and visualization.
- Write an inventory CSV and a download manifest CSV so the notebook shows exactly what was pulled.

QC checkpoint:
- The notebook should report a local `g25/` folder.
- The inventory table should include all requested approved subjects.
- The representative T1w, AdView BOLD, and AdView event files should exist locally after this cell runs.


In [ ]:
# Query the snapshot root first so we can resolve both the top-level metadata files
# and the subject-specific tree keys used later in the download loop.
root_rows = snapshot_files()
root_url_map = {row["filename"]: row["urls"][0] for row in root_rows if row.get("urls")}

# Pull dataset-level metadata before subject downloads. Later cells read these files
# to interpret task names, participant info, and BIDS-side context.
for filename in ["README", "dataset_description.json", "participants.tsv", "participants.json", "task-adview_bold.json", "task-SST_bold.json", "task-trust_bold.json"]:
    if filename in root_url_map:
        download_url(root_url_map[filename], BIDS_ROOT / filename, validate_nifti=False)
patch_task_metadata(ADVIEW_TASK_JSON)

download_records = []
inventory_records = []
representative_anat_path = None
representative_bold_path = None
representative_bold_json_path = None
representative_events_path = None

# Download loop per approved subject.
# The workflow keeps full anatomical coverage for preprocessing, but limits
# functional downloads to AdView-specific files so the notebook stays aligned
# with this task and avoids unnecessary SST/trust transfers.
for subject in APPROVED_SUBJECTS:
    subject_key = subject_directory_key(subject, root_rows)
    record = {"subject": subject}

    for dirname in ["anat", "func"]:
        try:
            dir_key = child_directory_key(subject_key, dirname)
            dir_rows = snapshot_files(dir_key)
            # All anatomical files are retained because fMRIPrep expects the full
            # structural context, not just a representative T1w.
            if dirname == "anat":
                wanted_rows = [row for row in dir_rows if row.get("urls")]
            else:
                wanted_rows = [
                    row for row in dir_rows
                    if row.get("urls") and f"_task-{TASK}_" in row["filename"]
                ]
            record[f"has_{dirname}_dir"] = True
            record[f"{dirname}_file_count"] = len(wanted_rows)
            record[f"{dirname}_bytes"] = sum(int(row.get("size") or 0) for row in wanted_rows)
            for row in wanted_rows:
                if not row.get("urls"):
                    continue
                destination = BIDS_ROOT / subject / dirname / row["filename"]
                validate = row["filename"].endswith(".nii.gz")
                download_url(row["urls"][0], destination, validate_nifti=validate)
                download_records.append(
                    {
                        "subject": subject,
                        "directory": dirname,
                        "filename": row["filename"],
                        "size_bytes": int(row.get("size") or 0),
                        "path": str(destination),
                    }
                )
        except FileNotFoundError:
            record[f"has_{dirname}_dir"] = False
            record[f"{dirname}_file_count"] = 0
            record[f"{dirname}_bytes"] = 0

    inventory_records.append(record)

anat_filename = f"{REPRESENTATIVE_SUBJECT}_T1w.nii.gz"
bold_filename = f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_echo-{ECHO}_part-mag_bold.nii.gz"
bold_json_filename = f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_echo-{ECHO}_part-mag_bold.json"
events_filename = f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_events.tsv"

representative_anat_path = BIDS_ROOT / REPRESENTATIVE_SUBJECT / "anat" / anat_filename
representative_bold_path = BIDS_ROOT / REPRESENTATIVE_SUBJECT / "func" / bold_filename
representative_bold_json_path = BIDS_ROOT / REPRESENTATIVE_SUBJECT / "func" / bold_json_filename
representative_events_path = BIDS_ROOT / REPRESENTATIVE_SUBJECT / "func" / events_filename

inventory = pd.DataFrame(inventory_records).sort_values("subject").reset_index(drop=True)
# Convert the raw bookkeeping lists into tables that the notebook can show inline
# and save for later auditing outside the notebook.
download_manifest = pd.DataFrame(download_records).sort_values(["subject", "directory", "filename"]).reset_index(drop=True)
inventory.to_csv(INVENTORY_PATH, index=False)
download_manifest.to_csv(DOWNLOAD_MANIFEST_PATH, index=False)

workspace_summary = {
    "project_root": str(PROJECT_ROOT),
    "approved_subjects": APPROVED_SUBJECTS,
    "approved_adview_subjects": APPROVED_ADVIEW_SUBJECTS,
    "representative_subject": REPRESENTATIVE_SUBJECT,
    "task": TASK,
    "run": RUN,
    "download_manifest": str(DOWNLOAD_MANIFEST_PATH),
}
WORKSPACE_SUMMARY_PATH.write_text(json.dumps(workspace_summary, indent=2) + "\n")

print(f"Inventory CSV: {INVENTORY_PATH}")
print(f"Download manifest CSV: {DOWNLOAD_MANIFEST_PATH}")
print(f"Downloaded files: {len(download_manifest)}")
print(f"Downloaded size (GiB): {download_manifest['size_bytes'].sum() / (1024 ** 3):.2f}")

display(inventory)
display(download_manifest.head(20))

# Promote the representative subject paths into simple variable names because the
# visualization, MCFLIRT fallback, and FEAT steps reuse them repeatedly.
anat_path = representative_anat_path
bold_path = representative_bold_path
bold_json_path = representative_bold_json_path
local_events_path = representative_events_path
bold_meta = json.loads(bold_json_path.read_text())

assert anat_path.exists()
assert bold_path.exists()
assert bold_json_path.exists()
assert local_events_path.exists()
assert set(APPROVED_ADVIEW_SUBJECTS).issubset(set(APPROVED_SUBJECTS))


## Step 2 - Inspect a representative anatomy file, AdView BOLD file, and AdView event structure

Purpose:
- Before preprocessing, confound extraction, or modeling, confirm that the representative imaging files open correctly and that the AdView event table has the categories we expect.

What the code is doing:
- Load the representative T1w and AdView BOLD files with `nibabel`.
- Save one anatomy figure and one middle-volume BOLD figure.
- Load the AdView `events.tsv` file.
- Derive the event labels that will later become FSL EV files.
- Save an event-count plot and a timeline plot.

QC checkpoint:
- The T1w image should be 3D.
- The AdView BOLD file should be 4D.
- The PNG figures should display inline below the cell.


In [ ]:
# Load one representative structural and functional image so the notebook can prove
# the downloaded files are readable before any preprocessing or modeling begins.
t1_img = nib.load(str(anat_path))
bold_img = nib.load(str(bold_path))
middle_bold_idx = bold_img.shape[3] // 2
middle_bold = np.asarray(bold_img.dataobj[..., middle_bold_idx])

# Translate the raw AdView events table into FEAT-facing condition labels. The same
# derived labels are reused later when Step 6 writes EV files.
events = pd.read_csv(local_events_path, sep="\t").copy()
events["derived_trial_type"] = events.apply(derive_adview_trial_type, axis=1)

# Figure paths live under output/jupyter-notebook/figures so the images become part
# of the saved notebook artifact set rather than temporary runtime files.
t1_png = FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_T1w.png"
bold_png = FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_middle-volume.png"
event_count_png = FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_event-counts.png"
event_timeline_png = FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_event-timeline.png"

print(f"T1w shape: {t1_img.shape}")
print(f"AdView BOLD shape: {bold_img.shape}")
print(f"Middle volume index: {middle_bold_idx}")
print(f"RepetitionTime from JSON: {bold_meta.get('RepetitionTime')}")
print("Derived AdView event counts:")
print(events["derived_trial_type"].value_counts().to_string())

assert len(t1_img.shape) == 3
assert len(bold_img.shape) == 4

# Save and immediately display the QC figures so the notebook remains readable both
# during interactive runs and when reopened later with stored outputs.
plot_orthogonal_slices(np.asarray(t1_img.dataobj), f"{REPRESENTATIVE_SUBJECT} T1w", t1_png)
plot_orthogonal_slices(middle_bold, f"{REPRESENTATIVE_SUBJECT} AdView run-{RUN} middle volume", bold_png)
plot_adview_events(events, event_count_png, event_timeline_png)

display(events.head(12))
display(Image(filename=str(t1_png)))
display(Image(filename=str(bold_png)))
display(Image(filename=str(event_count_png)))
display(Image(filename=str(event_timeline_png)))


## Step 3 - Attempt `fMRIPrep` across the approved subjects and record the run status

Purpose:
- The next confounds step needs `fMRIPrep` outputs.
- This cell attempts a real `fMRIPrep` run for each approved subject when the environment provides a runnable setup.
- If the environment cannot run `fMRIPrep`, the cell records a clean preflight blocker instead of failing silently.

What the code is doing:
- Run a `DRY_RUN=1` preflight through `code/fmriprep.sh` for each subject.
- Parse the detected execution mode.
- Reuse existing `fMRIPrep` outputs if they are already present.
- Otherwise, attempt a real run when `G25_RUN_FMRIPREP` is not disabled and the environment can actually run `fMRIPrep`.
- Save the status table to `g25/output/jupyter-notebook/g25-fmriprep-run-status.csv`.

QC checkpoint:
- `status = ran` or `status = reused-existing-output` means that subject has usable `fMRIPrep` output.
- `status = blocked` means the environment could not run `fMRIPrep`.
- `status = failed` means `fMRIPrep` started but did not finish successfully.


In [ ]:
# Record one row per subject so this cell becomes a status dashboard rather than
# a single all-or-nothing preprocessing attempt.
fmriprep_rows = []

for subject in APPROVED_SUBJECTS:
    subject_bids_dir = BIDS_ROOT / subject
    subject_func_dir = subject_bids_dir / "func"
    subject_has_bold = any(subject_func_dir.glob("*_bold.nii.gz"))
    # Preflight asks the helper script which execution mode is available in the
    # current environment before we attempt a real fMRIPrep run.
    preflight = run(
        ["bash", str(FMRIPREP_SCRIPT_PATH), subject.removeprefix("sub-")],
        cwd=PROJECT_ROOT,
        env={"DRY_RUN": "1"},
        check=False,
    )
    execution_mode = parse_stdout_value(preflight.stdout, "Execution mode")
    fmriprep_subject_dir = FMRIPREP_OUTDIR / subject
    existing_confounds = sorted(fmriprep_subject_dir.glob("func/*desc-confounds_timeseries.tsv"))
    row = {
        "subject": subject,
        "local_bids_present": subject_bids_dir.exists(),
        "local_bold_present": subject_has_bold,
        "execution_mode": execution_mode or "unknown",
        "preflight_returncode": preflight.returncode,
        "fmriprep_output_exists_before": bool(existing_confounds),
        "status": "preflight-only",
        "real_run_returncode": None,
        "fmriprep_output_exists_after": bool(existing_confounds),
        "blocker": "",
    }

    if not subject_has_bold:
        row["status"] = "missing-bold"
        row["blocker"] = "no local BOLD files downloaded for this subject"
        fmriprep_rows.append(row)
        continue

    # Reuse existing outputs when present so rerunning the notebook does not repeat
    # an expensive preprocessing job unnecessarily.
    if existing_confounds:
        row["status"] = "reused-existing-output"
        fmriprep_rows.append(row)
        continue

    if not RUN_REAL_FMRIPREP:
        row["status"] = "preflight-only"
        row["blocker"] = "RUN_REAL_FMRIPREP disabled via G25_RUN_FMRIPREP=0"
        fmriprep_rows.append(row)
        continue

    # Block cleanly when the environment cannot supply fMRIPrep. Later steps can
    # still continue through the MCFLIRT fallback path.
    if execution_mode == "unavailable":
        row["status"] = "blocked"
        row["blocker"] = "no runnable fMRIPrep setup detected"
        fmriprep_rows.append(row)
        continue

    real = run(
        ["bash", str(FMRIPREP_SCRIPT_PATH), subject.removeprefix("sub-")],
        cwd=PROJECT_ROOT,
        check=False,
    )
    row["real_run_returncode"] = real.returncode
    new_confounds = sorted(fmriprep_subject_dir.glob("func/*desc-confounds_timeseries.tsv"))
    row["fmriprep_output_exists_after"] = bool(new_confounds)
    if real.returncode == 0 and new_confounds:
        row["status"] = "ran"
    else:
        row["status"] = "failed"
        row["blocker"] = (real.stderr or real.stdout).splitlines()[-1] if (real.stderr or real.stdout) else "fMRIPrep failed without output"
    fmriprep_rows.append(row)

# Persist the per-subject status table so readers can inspect preprocessing state
# even outside the notebook UI.
fmriprep_status = pd.DataFrame(fmriprep_rows).sort_values("subject").reset_index(drop=True)
fmriprep_status.to_csv(FMRIPREP_STATUS_PATH, index=False)
display(fmriprep_status)
print(f"fMRIPrep status table saved to: {FMRIPREP_STATUS_PATH}")


## Step 4 - Extract FEAT-ready `fMRIPrep` confounds automatically

Purpose:
- `fMRIPrep` writes a rich confounds table, but FEAT usually needs a narrower nuisance-regressor subset.
- This cell runs the repo helper script directly from the notebook and records what was written vs what is still missing.

What the code is doing:
- Run `code/extract_fmriprep_confounds.py` across the current approved subject list for the AdView task and run 1 only.
- Read the generated manifest from `derivatives/fsl/confounds_fmriprep/confounds_manifest.csv`.
- Summarize which subject/task/run combinations produced FEAT-ready confounds.

QC checkpoint:
- `status = written` means a FEAT-ready confounds file was created.
- `status = missing` means the expected `fMRIPrep` confounds file does not exist yet for that subject/task/run.


In [ ]:
# Delegate confound selection to the existing helper script so the notebook uses the
# same logic as the standalone project scripts rather than a notebook-only variant.
extract_cmd = [
    sys.executable,
    str(EXTRACT_CONFOUNDS_PATH),
    "--project-root",
    str(PROJECT_ROOT),
    "--subjects",
    *APPROVED_SUBJECTS,
    "--tasks",
    TASK,
    "--runs",
    str(RUN),
]
# Run the extractor and surface stdout directly in the notebook. That keeps the
# confound-selection decisions visible instead of burying them in a hidden log.
extract_result = run(extract_cmd, cwd=PROJECT_ROOT, check=False)
print(extract_result.stdout)
if extract_result.returncode != 0:
    print(extract_result.stderr)
    raise RuntimeError("extract_fmriprep_confounds.py failed")

# The manifest is the canonical Step 4 output: it tells us which subject/task/run
# combinations produced selected confounds and which are still missing.
confounds_manifest_path = CONFOUNDS_FMRIPREP_DIR / "confounds_manifest.csv"
confounds_manifest = pd.read_csv(confounds_manifest_path)
confounds_summary = (
    confounds_manifest.groupby(["status", "task"]).size().rename("n_rows").reset_index().sort_values(["status", "task"])
)

display(confounds_summary)
display(confounds_manifest.head(20))
print(f"Confounds manifest: {confounds_manifest_path}")


## Step 5 - Choose the representative FEAT confound source

Purpose:
- The representative AdView FEAT model should use `fMRIPrep` confounds when they exist.
- If they do not exist yet, the notebook falls back to MCFLIRT motion confounds so the worked example can still run.

What the code is doing:
- Check whether the representative AdView `fMRIPrep` confounds file exists.
- If it exists, use it directly for FEAT.
- If it does not exist, run `mcflirt` on the representative AdView run and compute an FD QC plot as the fallback confounds source.

QC checkpoint:
- The cell should report whether the selected source is `fmriprep` or `mcflirt-fallback`.
- If the fallback path is used, the FD plot should appear inline below the cell.


In [ ]:
# FEAT needs one confound file for the representative run. Prefer the richer
# fMRIPrep-derived confounds when they exist; otherwise build a motion-only fallback.
representative_fmriprep_confounds = (
    CONFOUNDS_FMRIPREP_DIR
    / REPRESENTATIVE_SUBJECT
    / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_desc-fmriprepSelectedConfounds.tsv"
)

runtime_input_dir = RUNTIME_PROJECT / "output" / "jupyter-notebook" / "_cache" / "download-visualize" / REPRESENTATIVE_SUBJECT
runtime_feat_dir = RUNTIME_PROJECT / "output" / "jupyter-notebook" / "_runtime" / "feat-demo"
runtime_mc_dir = runtime_feat_dir / "mcflirt"
runtime_mc_dir.mkdir(parents=True, exist_ok=True)
CONF_DIR.mkdir(parents=True, exist_ok=True)

selected_confounds_for_feat = representative_fmriprep_confounds
selected_confounds_source = "fmriprep"
mcflirt_base = runtime_mc_dir / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_mcf"
mcflirt_par = mcflirt_base.with_suffix(".par")
confounds_path = CONF_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_mcflirt_confounds.tsv"
fd_metric_path = CONF_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_fd-metric.txt"
fd_plot_path = CONF_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_fd-plot.png"

# Fallback path: create FEAT-compatible nuisance regressors directly from MCFLIRT.
# This keeps the demo runnable even when Step 3 could not produce fMRIPrep outputs.
if not representative_fmriprep_confounds.exists():
    selected_confounds_source = "mcflirt-fallback"
    runtime_bold = ensure_uncompressed_nifti(
        bold_path,
        runtime_input_dir / bold_path.name.replace(".nii.gz", ".nii"),
    )
    fsl_ready, fsl_load_method = load_fsl_module_if_needed()
    mcflirt_cmd = find_first_command(["mcflirt"])
    if fsl_load_method and fsl_load_method != "already-on-path":
        print(f"FSL load method for this cell: {fsl_load_method}")

    if mcflirt_cmd is None:
        raise FileNotFoundError(
            "Could not find mcflirt on PATH, and automatic `ml fsl` / `module load fsl` did not succeed. "
            "Launch the notebook from a Neurodesk environment with FSL available."
        )

    # Reuse MCFLIRT outputs when possible, but clear stale partial files before a rerun
    # so FEAT does not inherit inconsistent motion parameters.
    if not mcflirt_par.exists() or not mcflirt_base.with_suffix(".nii.gz").exists():
        for stale_path in [mcflirt_par, mcflirt_base.with_suffix(".nii.gz")]:
            if stale_path.exists():
                stale_path.unlink()
        run([
            mcflirt_cmd,
            "-in", str(runtime_bold),
            "-out", str(mcflirt_base),
            "-plots",
            "-mats",
            "-report",
        ])

    confound_columns = ["rot_x", "rot_y", "rot_z", "trans_x", "trans_y", "trans_z"]
    confounds = pd.read_csv(mcflirt_par, sep=r"\s+", header=None, names=confound_columns)
    confounds.to_csv(confounds_path, sep="\t", index=False)

    # Framewise displacement is computed here from the rigid-body parameters so the
    # notebook can show a simple motion QC plot alongside the confounds table.
    motion_values = confounds.to_numpy(dtype=float)
    fd_values = np.zeros(motion_values.shape[0], dtype=float)
    if motion_values.shape[0] > 1:
        diffs = np.abs(np.diff(motion_values, axis=0))
        fd_values[1:] = (50.0 * diffs[:, :3].sum(axis=1)) + diffs[:, 3:].sum(axis=1)
    fd_metric = pd.DataFrame({"framewise_displacement": fd_values})
    fd_metric.to_csv(fd_metric_path, header=False, index=False)

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(fd_metric.index, fd_metric["framewise_displacement"], linewidth=1.2, color="steelblue")
    ax.set_title("Motion outlier metric: fd")
    ax.set_xlabel("time")
    ax.set_ylabel("metric value")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(fd_plot_path, dpi=150)
    plt.close(fig)

    selected_confounds_for_feat = mcflirt_par

print(f"Selected confounds source: {selected_confounds_source}")
print(f"Selected confounds path: {selected_confounds_for_feat}")
if representative_fmriprep_confounds.exists():
    display(pd.read_csv(representative_fmriprep_confounds, sep="\t", header=None).head())
else:
    display(pd.read_csv(confounds_path, sep="\t").head())
    display(Image(filename=str(fd_plot_path)))


## Step 6 - Convert all available AdView `events.tsv` files to FSL 3-column EV files

Purpose:
- This is the task-events to FEAT-EV conversion step for the AdView analysis.

What the code is doing:
- Loop over every approved subject that has AdView in `v1.0.0`.
- Derive event labels from `trial_type`, `is_attention_check`, and `attention_response`.
- Write FSL 3-column EV files into `g25/derivatives/fsl/EVfiles/sub-<id>/adview/`.
- Save a manifest and a per-subject summary CSV.

QC checkpoint:
- The manifest should include `sub-11028`, `sub-11039`, and `sub-11450`.
- Each EV file should have exactly three columns.


In [ ]:
# Build a manifest while writing EVs so the notebook can report both the count of
# files per subject and the exact path/row count for each condition file.
ev_manifest_rows = []
summary_rows = []
# Each approved AdView subject contributes run-1 events only in this snapshot.
# The helper function writes one 3-column file per modeled condition.
for subject in APPROVED_ADVIEW_SUBJECTS:
    subject_events = BIDS_ROOT / subject / "func" / f"{subject}_task-{TASK}_run-1_events.tsv"
    subject_ev_dir = EV_ROOT / subject / TASK
    manifest_rows = convert_adview_events_to_evs(subject_events, subject_ev_dir, run_id=1)
    summary_rows.append({"subject": subject, "run": 1, "n_ev_files": len(manifest_rows)})
    for row in manifest_rows:
        ev_manifest_rows.append({
            "subject": subject,
            "run": 1,
            "event": row["event"],
            "rows": row["rows"],
            "path": row["path"],
        })

ev_summary = pd.DataFrame(summary_rows).sort_values(["subject", "run"]).reset_index(drop=True)
ev_manifest = pd.DataFrame(ev_manifest_rows).sort_values(["subject", "run", "event"]).reset_index(drop=True)
# Save both a compact summary and a detailed manifest so downstream steps and human
# readers can inspect the EV conversion without opening every text file manually.
all_ev_summary_path = OUTPUT_DIR / "g25-adview-run-summary.csv"
all_ev_manifest_path = OUTPUT_DIR / "g25-adview-ev-manifest.csv"
ev_summary.to_csv(all_ev_summary_path, index=False)
ev_manifest.to_csv(all_ev_manifest_path, index=False)

print(f"AdView run summary: {all_ev_summary_path}")
print(f"AdView EV manifest: {all_ev_manifest_path}")
display(ev_summary)
display(ev_manifest)


## Step 7 - Run one representative first-level FEAT model for AdView

Purpose:
- This creates the final worked example from BOLD data, confounds, and EV files to a FEAT first-level result.

What the code is doing:
- Build a subject/run-specific `.fsf` file from the local AdView FEAT template.
- Point FEAT at the chosen confounds source, the generated EV files, and the representative BOLD image.
- Run `feat` and copy the important outputs into `g25/output/jupyter-notebook/feat-demo/`.

QC checkpoint:
- The FEAT design matrix figures should appear inline.
- `zstat1.nii.gz` should exist in the demo output folder.


In [ ]:
# FEAT runs inside a runtime directory, but key outputs are copied back into
# output/jupyter-notebook/feat-demo so the notebook can display stable artifact paths.
runtime_feat_output = RUNTIME_PROJECT / "output" / "jupyter-notebook" / "_runtime" / "feat-demo" / "feat" / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_model-1_demo"
runtime_feat_output.parent.mkdir(parents=True, exist_ok=True)
runtime_fsf_path = runtime_feat_output.parent / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_model-1_demo.fsf"
runtime_feat_output_dir = Path(str(runtime_feat_output) + ".feat")

feat_demo_files = {
    "fsf": FEAT_DEMO_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_model-1_demo.fsf",
    "design": FEAT_DEMO_DIR / "design.png",
    "design_cov": FEAT_DEMO_DIR / "design_cov.png",
    "example_func": FEAT_DEMO_DIR / "example_func.nii.gz",
    "zstat1": FEAT_DEMO_DIR / "zstat1.nii.gz",
    "report": FEAT_DEMO_DIR / "report.html",
    "report_log": FEAT_DEMO_DIR / "report_log.html",
    "feat1_log": FEAT_DEMO_DIR / "feat1.log",
}

# Only build the representative model if the final zstat is missing. This keeps the
# notebook rerunnable without paying the FEAT cost every time.
if not feat_demo_files["zstat1"].exists():
    missed_trial_path = RUNTIME_PROJECT / "derivatives" / "fsl" / "EVfiles" / REPRESENTATIVE_SUBJECT / TASK / "run-1_missed_trial.txt"
    ev_shape = "3" if missed_trial_path.exists() else "10"
    bold_for_feat = mcflirt_base.with_suffix(".nii.gz") if selected_confounds_source == "mcflirt-fallback" else ensure_uncompressed_nifti(bold_path, runtime_input_dir / bold_path.name.replace(".nii.gz", ".nii"))
    # The FEAT design file is generated from the AdView template by substituting the
    # runtime data path, EV directory, smoothing value, and chosen confound file.
    template_text = ADVIEW_TEMPLATE_PATH.read_text()
    fsf_text = (
        template_text
        .replace("OUTPUT", str(runtime_feat_output))
        .replace("DATA", str(bold_for_feat))
        .replace("NVOLS", str(bold_img.shape[3]))
        .replace("EVDIR", str((RUNTIME_PROJECT / "derivatives" / "fsl" / "EVfiles" / REPRESENTATIVE_SUBJECT / TASK / "run-1")))
        .replace("MISSED_TRIAL", str(missed_trial_path))
        .replace("EV_SHAPE", ev_shape)
        .replace("SMOOTH", "5")
        .replace("CONFOUNDEVS", str(selected_confounds_for_feat))
    )
    runtime_fsf_path.write_text(fsf_text)

    feat_ready, feat_load_method = load_fsl_module_if_needed()
    feat_cmd = find_first_command(["feat"])
    if feat_load_method and feat_load_method != "already-on-path":
        print(f"FSL load method for FEAT cell: {feat_load_method}")
    if feat_cmd is None:
        raise FileNotFoundError(
            "Could not find feat on PATH, and automatic `ml fsl` / `module load fsl` did not succeed. "
            "Launch the notebook from a Neurodesk environment with FSL available."
        )
    run([feat_cmd, str(runtime_fsf_path)], cwd=runtime_feat_output.parent)

    # Copy the important FEAT artifacts out of the runtime tree so the saved notebook
    # always points at stable, reader-facing output locations.
    shutil.copy2(runtime_fsf_path, feat_demo_files["fsf"])
    shutil.copy2(runtime_feat_output_dir / "design.png", feat_demo_files["design"])
    shutil.copy2(runtime_feat_output_dir / "design_cov.png", feat_demo_files["design_cov"])
    shutil.copy2(runtime_feat_output_dir / "example_func.nii.gz", feat_demo_files["example_func"])
    shutil.copy2(runtime_feat_output_dir / "stats" / "zstat1.nii.gz", feat_demo_files["zstat1"])
    shutil.copy2(runtime_feat_output_dir / "report.html", feat_demo_files["report"])
    shutil.copy2(runtime_feat_output_dir / "report_log.html", feat_demo_files["report_log"])
    shutil.copy2(runtime_feat_output_dir / "logs" / "feat1", feat_demo_files["feat1_log"])
    feat_mode = "ran-feat-now"
else:
    feat_mode = "reused-cached-feat-demo"

print(f"FEAT mode: {feat_mode}")
print(f"FSF file: {feat_demo_files['fsf']}")
print(f"Example func: {feat_demo_files['example_func']}")
print(f"zstat1: {feat_demo_files['zstat1']}")

display(Image(filename=str(feat_demo_files["design"])))
display(Image(filename=str(feat_demo_files["design_cov"])))


## Step 8 - Visualize the AdView FEAT result with MRIcroGL and embed the capture

Purpose:
- A saved FEAT result is easier to inspect when the activation map is rendered on top of the representative functional image.

What the code is doing:
- Load the AdView `zstat1` image.
- Find the peak voxel in the statistic map.
- Build a short MRIcroGL script that centers the view on that peak.
- Save the screenshot into the notebook output folder and display it inline.

QC checkpoint:
- The MRIcroGL script text should appear inline.
- If `mricrogl` is available on PATH, the screenshot should display inline below the cell.


In [ ]:
# Load the representative FEAT statistic map and find the strongest voxel so the
# MRIcroGL view opens at a meaningful location instead of an arbitrary coordinate.
zstat_img = nib.load(str(feat_demo_files["zstat1"]))
zstat_data = zstat_img.get_fdata()
peak_idx = np.unravel_index(np.nanargmax(np.abs(zstat_data)), zstat_data.shape)
peak_mm = nib.affines.apply_affine(zstat_img.affine, peak_idx)

mricrogl_png = FEAT_FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_zstat1_mricrogl.png"
runtime_example_func = RUNTIME_PROJECT / "output" / "jupyter-notebook" / "feat-demo" / "example_func.nii.gz"
runtime_zstat1 = RUNTIME_PROJECT / "output" / "jupyter-notebook" / "feat-demo" / "zstat1.nii.gz"

# Build the MRIcroGL script as plain text. Keeping it visible inside the notebook
# makes the rendering step easier to debug and easier for readers to adapt.
mricrogl_script = textwrap.dedent(f'''\
import gl
gl.resetdefaults()
gl.loadimage({str(runtime_example_func)!r})
gl.overlayload({str(runtime_zstat1)!r})
gl.minmax(1, 2.5, 8)
gl.opacity(1, 80)
gl.colorname(1, '1red')
gl.orthoviewmm({peak_mm[0]:.3f}, {peak_mm[1]:.3f}, {peak_mm[2]:.3f})
gl.savebmp({str(mricrogl_png)!r})
gl.quit()
''')

print(f"Peak voxel index: {peak_idx}")
print(f"Peak millimeter coordinates: {tuple(round(v, 3) for v in peak_mm)}")
display(Markdown("**MRIcroGL Script**"))
display(Markdown(f"```python\n{mricrogl_script}```"))

# MRIcroGL is optional at notebook-authoring time but expected in Neurodesk. The
# conditional keeps the notebook readable on systems where the viewer is absent.
if MGL_CMD is None:
    print("SKIP: MRIcroGL command not found on PATH in this environment. This step is expected to run in Neurodesk where mricrogl is available.")
else:
    rendered_path, rendered_with, optional_mricron = render_with_mricrogl(mricrogl_script, mricrogl_png)
    print(f"Rendered with: {rendered_with}")
    if optional_mricron:
        print(f"MRIcron also available at: {optional_mricron}")
    print(f"MRIcroGL screenshot: {rendered_path}")

if mricrogl_png.exists() and mricrogl_png.stat().st_size > 0:
    display(Image(filename=str(mricrogl_png)))


## Troubleshooting and summary

This notebook is written to be standalone, but two steps are environment-dependent:
- the `fMRIPrep` step needs either a direct `fmriprep` command, a Neurodesk `ml fmriprep` module, or a container runtime; for the FreeSurfer license it checks `FS_LICENSE_FILE`, `~/.license`, `~/license.txt`, and `~/Desktop/license.txt`
- the FEAT/MRIcroGL steps need FSL and MRIcroGL available in the runtime environment

If you only want a fast validation run during development, launch Jupyter with:

```bash
G25_SUBJECTS=sub-11039 G25_RUN_FMRIPREP=0 jupyter lab g25-new-AdView-datalad-confounds-evs-feat-mricrogl.ipynb
```


In [ ]:
# Collect the main persistent artifacts in one table so someone reading the notebook
# can map each workflow step to the files it produced or reused.
summary_rows = [
    {"artifact": "Workspace summary JSON", "path": str(WORKSPACE_SUMMARY_PATH)},
    {"artifact": "Approved-subject inventory CSV", "path": str(INVENTORY_PATH)},
    {"artifact": "Approved-subject download manifest CSV", "path": str(DOWNLOAD_MANIFEST_PATH)},
    {"artifact": "fMRIPrep status CSV", "path": str(FMRIPREP_STATUS_PATH)},
    {"artifact": "fMRIPrep confounds manifest", "path": str(CONFOUNDS_FMRIPREP_DIR / "confounds_manifest.csv")},
    {"artifact": "Representative T1w PNG", "path": str(FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_T1w.png")},
    {"artifact": "Representative AdView BOLD PNG", "path": str(FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_middle-volume.png")},
    {"artifact": "AdView event-count PNG", "path": str(FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_event-counts.png")},
    {"artifact": "AdView event timeline PNG", "path": str(FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_event-timeline.png")},
    {"artifact": "Representative FEAT confounds source", "path": str(selected_confounds_for_feat)},
    {"artifact": "All-subject AdView EV summary", "path": str(OUTPUT_DIR / "g25-adview-run-summary.csv")},
    {"artifact": "All-subject AdView EV manifest", "path": str(OUTPUT_DIR / "g25-adview-ev-manifest.csv")},
    {"artifact": "FEAT FSF file", "path": str(feat_demo_files["fsf"])},
    {"artifact": "FEAT zstat1", "path": str(feat_demo_files["zstat1"])},
]
if "mricrogl_png" in globals() and mricrogl_png.exists():
    summary_rows.append({"artifact": "MRIcroGL screenshot", "path": str(mricrogl_png)})
summary = pd.DataFrame(summary_rows)
display(summary)
